In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.trial import TrialState
import json

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
WAVLM_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_wavlm_only.pkl")
WAV2VEC_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_with_text.pkl")

# 모델 기본 설정
WAVLM_DIM = 768
WAV2VEC_DIM = 768
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_ENHANCED_DIM = 6

# 학습 기본 설정
BATCH_SIZE = 8
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Optuna 설정
N_TRIALS = 100
STUDY_NAME = "multimodal_wavlm_wav2vec_balanced_f1"
OPTUNA_EPOCHS = 20


# =============================================================================
# Enhanced TTR Feature Extraction
# =============================================================================
def extract_enhanced_ttr_features(text):
    """TTR 관련 강화된 특징 추출"""
    if not text or len(text.strip()) == 0:
        return {
            'ttr': 0.0,
            'ttr_log': 0.0,
            'repetition_rate': 0.0,
            'unique_word_ratio': 0.0,
            'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    tokens = text.lower().split()
    
    if len(tokens) == 0:
        return {
            'ttr': 0.0,
            'ttr_log': 0.0,
            'repetition_rate': 0.0,
            'unique_word_ratio': 0.0,
            'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    unique_tokens = set(tokens)
    ttr = len(unique_tokens) / len(tokens)
    ttr_log = len(unique_tokens) / math.log(len(tokens) + 1)
    
    word_counts = {}
    for token in tokens:
        word_counts[token] = word_counts.get(token, 0) + 1
    
    repeated_words = sum(1 for count in word_counts.values() if count > 1)
    repetition_rate = repeated_words / len(unique_tokens) if len(unique_tokens) > 0 else 0.0
    
    unique_words = sum(1 for count in word_counts.values() if count == 1)
    unique_word_ratio = unique_words / len(tokens)
    
    function_words = {
    'the', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'up', 'about', 'into', 'through', 'during',
    'he', 'she', 'it', 'we', 'they', 'him', 'her', 'us', 'them',
    'my', 'your', 'his', 'her', 'its', 'our', 'their',
    'am', 'is', 'are', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could',
    'this', 'that', 'these', 'those',
    'what', 'which', 'who', 'when', 'where', 'why', 'how'}
    
    content_words = [token for token in tokens if token not in function_words]
    lexical_density = len(content_words) / len(tokens) if len(tokens) > 0 else 0.0
    
    word_lengths = [len(token) for token in tokens]
    word_length_variance = np.var(word_lengths) if len(word_lengths) > 1 else 0.0
    
    return {
        'ttr': ttr,
        'ttr_log': ttr_log,
        'repetition_rate': repetition_rate,
        'unique_word_ratio': unique_word_ratio,
        'lexical_density': lexical_density,
        'word_length_variance': word_length_variance
    }


# =============================================================================
# Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# Multimodal Fusion Transformer Model
# =============================================================================
class MultimodalWavLMWav2VecTransformer(nn.Module):
    def __init__(
        self,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        dim_feedforward=512,
        dropout=0.3,
        fusion_strategy='adaptive',  # 'early', 'late', 'adaptive'
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(MultimodalWavLMWav2VecTransformer, self).__init__()
        
        self.d_model = d_model
        self.fusion_strategy = fusion_strategy
        
        # Q-type embedding
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # TTR projection
        self.ttr_projection = nn.Sequential(
            nn.Linear(TTR_ENHANCED_DIM, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32)
        )
        
        # === Fusion Strategy ===
        if fusion_strategy == 'early':
            # Early Fusion: Concatenate WavLM + Wav2Vec
            input_dim = WAVLM_DIM + WAV2VEC_DIM + q_type_embed_dim + 32
            self.input_projection = nn.Linear(input_dim, d_model)
            
        elif fusion_strategy == 'late':
            # Late Fusion: Separate branches
            self.wavlm_branch = nn.Sequential(
                nn.Linear(WAVLM_DIM, d_model // 2),
                nn.LayerNorm(d_model // 2),
                nn.GELU(),
                nn.Dropout(dropout)
            )
            
            self.wav2vec_branch = nn.Sequential(
                nn.Linear(WAV2VEC_DIM, d_model // 2),
                nn.LayerNorm(d_model // 2),
                nn.GELU(),
                nn.Dropout(dropout)
            )
            
            # Context features
            context_dim = q_type_embed_dim + 32
            self.context_projection = nn.Linear(context_dim, d_model // 4)
            
            # Final projection
            fusion_dim = d_model // 2 + d_model // 2 + d_model // 4
            self.input_projection = nn.Linear(fusion_dim, d_model)
            
        elif fusion_strategy == 'adaptive':
            # Adaptive Fusion: Learned gating
            self.wavlm_projection = nn.Linear(WAVLM_DIM, d_model)
            self.wav2vec_projection = nn.Linear(WAV2VEC_DIM, d_model)
            
            # Gating network
            gate_input_dim = WAVLM_DIM + WAV2VEC_DIM + q_type_embed_dim + 32
            self.gate_network = nn.Sequential(
                nn.Linear(gate_input_dim, 128),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(128, 2),  # 2 gates for WavLM and Wav2Vec
                nn.Softmax(dim=-1)
            )
            
            # Context projection
            context_dim = q_type_embed_dim + 32
            self.context_projection = nn.Linear(context_dim, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wavlm, batch_wav2vec, batch_ttr_enhanced, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_wavlm.device
        
        # Embeddings
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttr_projected = self.ttr_projection(batch_ttr_enhanced)
        
        # === Fusion ===
        if self.fusion_strategy == 'early':
            # Early Fusion: Simple concatenation
            combined_features = torch.cat([
                batch_wavlm,
                batch_wav2vec,
                q_type_embs,
                ttr_projected
            ], dim=1)
            
        elif self.fusion_strategy == 'late':
            # Late Fusion: Separate processing then combine
            wavlm_feat = self.wavlm_branch(batch_wavlm)
            wav2vec_feat = self.wav2vec_branch(batch_wav2vec)
            context_feat = self.context_projection(torch.cat([q_type_embs, ttr_projected], dim=1))
            
            combined_features = torch.cat([
                wavlm_feat,
                wav2vec_feat,
                context_feat
            ], dim=1)
            
        elif self.fusion_strategy == 'adaptive':
            # Adaptive Fusion: Learned gating
            wavlm_feat = self.wavlm_projection(batch_wavlm)
            wav2vec_feat = self.wav2vec_projection(batch_wav2vec)
            
            # Compute gates
            gate_input = torch.cat([batch_wavlm, batch_wav2vec, q_type_embs, ttr_projected], dim=1)
            gates = self.gate_network(gate_input)  # [N, 2]
            
            # Apply gates
            wavlm_gated = wavlm_feat * gates[:, 0:1]
            wav2vec_gated = wav2vec_feat * gates[:, 1:2]
            
            # Combine
            audio_fused = wavlm_gated + wav2vec_gated
            context_feat = self.context_projection(torch.cat([q_type_embs, ttr_projected], dim=1))
            
            combined_features = audio_fused + context_feat
            
            # Store gates for visualization
            self.last_gates = gates
        
        # Split by participants
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            
            if self.fusion_strategy in ['early', 'late']:
                participant_sequences.append(combined_features[start_idx:end_idx])
            else:  # adaptive
                participant_sequences.append(combined_features[start_idx:end_idx])
            
            start_idx = end_idx
        
        # Padding
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding_size = max_num_utterances - num_utts
                
                if self.fusion_strategy in ['early', 'late']:
                    padding = torch.zeros(padding_size, seq.size(1), device=device)
                else:  # adaptive
                    padding = torch.zeros(padding_size, self.d_model, device=device)
                
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        # Project to d_model
        if self.fusion_strategy in ['early', 'late']:
            x = self.input_projection(padded_sequences)
        else:  # adaptive - already in d_model
            x = padded_sequences
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        # Extend mask for CLS token
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional encoding
        x = self.pos_encoder(x)
        
        # Transformer encoding
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # CLS output
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        # Attention weights
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate
# =============================================================================
class MultimodalUtteranceDataset(Dataset):
    def __init__(self, wavlm_data, wav2vec_data):
        """
        wavlm_data: {pid: {'label': ..., 'utterances': [...]}}
        wav2vec_data: {pid: {'label': ..., 'utterances': [...]}}
        """
        self.data = []
        
        # 두 데이터셋의 공통 PID만 사용
        common_pids = set(wavlm_data.keys()) & set(wav2vec_data.keys())
        
        for pid in common_pids:
            # 발화 수가 같은지 확인
            if wavlm_data[pid]['num_utterances'] != wav2vec_data[pid]['num_utterances']:
                print(f"⚠️  Warning: PID {pid} has different utterance counts")
                continue
            
            self.data.append({
                'pid': pid,
                'label': wavlm_data[pid]['label'],
                'wavlm_utterances': wavlm_data[pid]['utterances'],
                'wav2vec_utterances': wav2vec_data[pid]['utterances'],
                'num_utterances': wavlm_data[pid]['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def multimodal_collate_fn(batch):
    batch_labels = []
    all_wavlm_utterances = []
    all_wav2vec_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_wavlm_utterances.extend(item['wavlm_utterances'])
        all_wav2vec_utterances.extend(item['wav2vec_utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    batch_wavlm = []
    batch_wav2vec = []
    batch_ttr_enhanced = []
    batch_q_type_ids = []
    
    for wavlm_utt, wav2vec_utt in zip(all_wavlm_utterances, all_wav2vec_utterances):
        batch_wavlm.append(wavlm_utt['wavlm'])
        batch_wav2vec.append(wav2vec_utt['wav2vec'])
        batch_q_type_ids.append(wavlm_utt['q_type_id'])
        
        text = wavlm_utt.get('text', '')
        ttr_features = extract_enhanced_ttr_features(text)
        
        ttr_vector = [
            ttr_features['ttr'],
            ttr_features['ttr_log'],
            ttr_features['repetition_rate'],
            ttr_features['unique_word_ratio'],
            ttr_features['lexical_density'],
            ttr_features['word_length_variance']
        ]
        batch_ttr_enhanced.append(ttr_vector)
    
    batch_wavlm = torch.FloatTensor(np.array(batch_wavlm))
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttr_enhanced = torch.FloatTensor(np.array(batch_ttr_enhanced))
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    return {
        'batch_wavlm': batch_wavlm,
        'batch_wav2vec': batch_wav2vec,
        'batch_ttr_enhanced': batch_ttr_enhanced,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }


# =============================================================================
# 데이터 로드
# =============================================================================
def load_and_split_multimodal_data():
    print(f"{'='*70}")
    print(f"📂 Multimodal 데이터 로드 중... (WavLM + Wav2Vec)")
    print(f"{'='*70}")
    
    # WavLM 데이터 로드
    print(f"Loading WavLM data...")
    with open(WAVLM_DATA_PATH, 'rb') as f:
        wavlm_dataset = pickle.load(f)
    
    # Wav2Vec 데이터 로드
    print(f"Loading Wav2Vec data...")
    with open(WAV2VEC_DATA_PATH, 'rb') as f:
        wav2vec_dataset = pickle.load(f)
    
    # 메타데이터
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    # 공통 PID 추출
    common_pids = set(wavlm_dataset.keys()) & set(wav2vec_dataset.keys())
    
    train_pids = [pid for pid in meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist() 
                  if pid in common_pids]
    val_pids = [pid for pid in meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
                if pid in common_pids]
    test_pids = [pid for pid in meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
                 if pid in common_pids]
    
    train_labels = [wavlm_dataset[pid]['label'] for pid in train_pids]
    val_labels = [wavlm_dataset[pid]['label'] for pid in val_pids]
    test_labels = [wavlm_dataset[pid]['label'] for pid in test_pids]
    
    print(f"\n공통 참가자 수: {len(common_pids)}")
    print(f"  - 정상 (0): {sum(1 for pid in common_pids if wavlm_dataset[pid]['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for pid in common_pids if wavlm_dataset[pid]['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    # 데이터셋 생성
    train_wavlm = {pid: wavlm_dataset[pid] for pid in train_pids}
    train_wav2vec = {pid: wav2vec_dataset[pid] for pid in train_pids}
    
    val_wavlm = {pid: wavlm_dataset[pid] for pid in val_pids}
    val_wav2vec = {pid: wav2vec_dataset[pid] for pid in val_pids}
    
    test_wavlm = {pid: wavlm_dataset[pid] for pid in test_pids}
    test_wav2vec = {pid: wav2vec_dataset[pid] for pid in test_pids}
    
    train_dataset = MultimodalUtteranceDataset(train_wavlm, train_wav2vec)
    val_dataset = MultimodalUtteranceDataset(val_wavlm, val_wav2vec)
    test_dataset = MultimodalUtteranceDataset(test_wavlm, test_wav2vec)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                             collate_fn=multimodal_collate_fn, num_workers=0,
                             pin_memory=True if torch.cuda.is_available() else False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=multimodal_collate_fn, num_workers=0,
                           pin_memory=True if torch.cuda.is_available() else False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=multimodal_collate_fn, num_workers=0,
                            pin_memory=True if torch.cuda.is_available() else False)
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 함수
# =============================================================================
def evaluate_multimodal(model, dataloader, criterion, threshold=0.5):
    """평가 함수"""
    model.eval()
    
    all_labels = []
    all_probs = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_wavlm,
                batch_wav2vec,
                batch_ttr_enhanced,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader)
    all_preds = (np.array(all_probs) > threshold).astype(int)
    
    overall_f1 = f1_score(all_labels, all_preds, average='binary')
    f1_per_class = f1_score(all_labels, all_preds, average=None)
    f1_normal = f1_per_class[0]
    f1_depression = f1_per_class[1]
    
    if f1_normal > 0 and f1_depression > 0:
        balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
    else:
        balanced_f1 = 0.0
    
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    tn = sum((l == 0 and p == 0) for l, p in zip(all_labels, all_preds))
    fp = sum((l == 0 and p == 1) for l, p in zip(all_labels, all_preds))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return {
        'loss': avg_loss,
        'overall_f1': overall_f1,
        'balanced_f1': balanced_f1,
        'f1_normal': f1_normal,
        'f1_depression': f1_depression,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'labels': all_labels,
        'probs': all_probs,
        'preds': all_preds
    }


def find_optimal_threshold(labels, probs, metric='balanced_f1'):
    """최적 threshold 찾기"""
    best_score = 0.0
    best_threshold = 0.5
    best_metrics = {
        'threshold': 0.5,
        'balanced_f1': 0.0,
        'overall_f1': 0.0,
        'f1_normal': 0.0,
        'f1_depression': 0.0,
        'precision': 0.0,
        'recall': 0.0
    }
    
    for thresh in np.arange(0.1, 0.9, 0.01):
        preds = (np.array(probs) > thresh).astype(int)
        
        if len(np.unique(preds)) < 2:
            continue
        
        overall_f1 = f1_score(labels, preds, average='binary', zero_division=0)
        f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
        
        if len(f1_per_class) < 2:
            continue
            
        f1_normal = f1_per_class[0]
        f1_depression = f1_per_class[1]
        
        if f1_normal > 0 and f1_depression > 0:
            balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
        else:
            balanced_f1 = 0.0
        
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        if metric == 'balanced_f1':
            score = balanced_f1
        elif metric == 'overall_f1':
            score = overall_f1
        elif metric == 'recall':
            score = recall
        else:
            score = balanced_f1
        
        if score > best_score:
            best_score = score
            best_threshold = thresh
            best_metrics = {
                'threshold': thresh,
                'balanced_f1': balanced_f1,
                'overall_f1': overall_f1,
                'f1_normal': f1_normal,
                'f1_depression': f1_depression,
                'precision': precision,
                'recall': recall
            }
    
    return best_threshold, best_metrics


# =============================================================================
# Training (간소화 버전)
# =============================================================================
def train_multimodal_model(fusion_strategy='adaptive'):
    """Multimodal 모델 학습"""
    print(f"\n{'='*70}")
    print(f"🚀 Multimodal 학습 시작 (Fusion: {fusion_strategy})")
    print(f"{'='*70}\n")
    
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_multimodal_data()
    
    # 모델 생성 (기본 하이퍼파라미터)
    model = MultimodalWavLMWav2VecTransformer(
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        dim_feedforward=512,
        dropout=0.3,
        fusion_strategy=fusion_strategy
    ).to(DEVICE)
    
    criterion = FocalLoss(alpha=0.6, gamma=2.0, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    
    best_balanced_f1 = 0.0
    best_threshold = 0.5
    patience_counter = 0
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_balanced_f1': [],
        'val_overall_f1': [],
        'val_f1_normal': [],
        'val_f1_depression': [],
        'val_precision': [],
        'val_recall': []
    }
    
    for epoch in range(NUM_EPOCHS):
        # Training
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
        for batch in pbar:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wavlm,
                batch_wav2vec,
                batch_ttr_enhanced,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        val_results = evaluate_multimodal(model, val_loader, criterion, threshold=0.5)
        opt_threshold, threshold_metrics = find_optimal_threshold(
            val_results['labels'],
            val_results['probs'],
            metric='balanced_f1'
        )
        
        # History 저장
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_results['loss'])
        history['val_balanced_f1'].append(threshold_metrics['balanced_f1'])
        history['val_overall_f1'].append(threshold_metrics['overall_f1'])
        history['val_f1_normal'].append(threshold_metrics['f1_normal'])
        history['val_f1_depression'].append(threshold_metrics['f1_depression'])
        history['val_precision'].append(threshold_metrics['precision'])
        history['val_recall'].append(threshold_metrics['recall'])
        
        scheduler.step()
        
        # 출력
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {val_results['loss']:.4f}")
        print(f"  Optimal Threshold: {opt_threshold:.3f}")
        print(f"  Balanced F1: {threshold_metrics['balanced_f1']:.4f} ⭐")
        print(f"  Overall F1: {threshold_metrics['overall_f1']:.4f}")
        print(f"  F1 Normal (0): {threshold_metrics['f1_normal']:.4f}")
        print(f"  F1 Depression (1): {threshold_metrics['f1_depression']:.4f}")
        print(f"  Precision: {threshold_metrics['precision']:.4f}")
        print(f"  Recall: {threshold_metrics['recall']:.4f}")
        
        # Best model 저장
        if threshold_metrics['balanced_f1'] > best_balanced_f1:
            best_balanced_f1 = threshold_metrics['balanced_f1']
            best_threshold = opt_threshold
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'fusion_strategy': fusion_strategy,
                'balanced_f1': best_balanced_f1,
                'optimal_threshold': best_threshold,
                'history': history,
                'threshold_metrics': threshold_metrics
            }, os.path.join(BASE_PATH, f'best_multimodal_{fusion_strategy}_model.pt'))
            
            print(f"  ✅ Best model saved! (Balanced F1: {best_balanced_f1:.4f})")
        else:
            patience_counter += 1
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping!")
                break
        
        print("-" * 70)
    
    # Test Set 평가
    print(f"\n{'='*70}")
    print(f"📊 Test Set 평가")
    print(f"{'='*70}")
    
    model_path = os.path.join(BASE_PATH, f'best_multimodal_{fusion_strategy}_model.pt')
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        best_threshold = checkpoint['optimal_threshold']
        
        test_results = evaluate_multimodal(model, test_loader, nn.BCEWithLogitsLoss(), threshold=best_threshold)
        
        print(f"\n🎯 Test Set Results:")
        print(f"  Balanced F1: {test_results['balanced_f1']:.4f} ⭐")
        print(f"  Overall F1: {test_results['overall_f1']:.4f}")
        print(f"  F1 Normal (0): {test_results['f1_normal']:.4f}")
        print(f"  F1 Depression (1): {test_results['f1_depression']:.4f}")
        print(f"  Precision: {test_results['precision']:.4f}")
        print(f"  Recall: {test_results['recall']:.4f}")
        print(f"  Specificity: {test_results['specificity']:.4f}")
        
        print(f"\n{'='*70}")
        print("분류 보고서:")
        print(f"{'='*70}")
        print(classification_report(
            test_results['labels'],
            test_results['preds'],
            target_names=['Normal', 'Depression'],
            digits=4
        ))
    
    return model, history, best_threshold


# =============================================================================
# 메인
# =============================================================================
if __name__ == "__main__":
    print(f"\n{'='*70}")
    print(f"🤖 Multimodal WavLM + Wav2Vec Transformer")
    print(f"{'='*70}\n")
    
    # 3가지 Fusion 전략 테스트
    fusion_strategies = ['early', 'late', 'adaptive']
    results_summary = {}
    
    for strategy in fusion_strategies:
        print(f"\n{'='*70}")
        print(f"Testing Fusion Strategy: {strategy.upper()}")
        print(f"{'='*70}\n")
        
        model, history, best_threshold = train_multimodal_model(fusion_strategy=strategy)
        
        # 결과 저장
        results_summary[strategy] = {
            'best_balanced_f1': max(history['val_balanced_f1']),
            'best_threshold': best_threshold
        }
    
    # 최종 비교
    print(f"\n{'='*70}")
    print(f"📊 Fusion Strategy 비교 결과")
    print(f"{'='*70}\n")
    
    for strategy, results in results_summary.items():
        print(f"{strategy.upper():10s}: Balanced F1 = {results['best_balanced_f1']:.4f}, Threshold = {results['best_threshold']:.3f}")
    
    print(f"\n{'='*70}")
    print(f"✅ 모든 작업 완료!")
    print(f"{'='*70}\n")

c:\Users\Lenovo\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (



🤖 Multimodal WavLM + Wav2Vec Transformer


Testing Fusion Strategy: EARLY


🚀 Multimodal 학습 시작 (Fusion: early)

📂 Multimodal 데이터 로드 중... (WavLM + Wav2Vec)
Loading WavLM data...
Loading Wav2Vec data...

공통 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)


Epoch 1/50:   0%|          | 0/14 [00:00<?, ?it/s]c:\Users\Lenovo\anaconda3\lib\site-packages\torch\nn\functional.py:5504: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
Epoch 1/50: 100%|██████████| 14/14 [00:01<00:00,  8.57it/s, loss=0.0566]
c:\Users\Lenovo\anaconda3\lib\site-packages\torch\nn\modules\transformer.py:408: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ..\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)



Epoch 1/50
  Train Loss: 0.0791
  Val Loss: 0.0872
  Optimal Threshold: 0.470
  Balanced F1: 0.5424 ⭐
  Overall F1: 0.5714
  F1 Normal (0): 0.5161
  F1 Depression (1): 0.5714
  Precision: 0.4348
  Recall: 0.8333
  ✅ Best model saved! (Balanced F1: 0.5424)
----------------------------------------------------------------------


Epoch 2/50: 100%|██████████| 14/14 [00:00<00:00, 14.13it/s, loss=0.072] 


...


Epoch 23/50
  Train Loss: 0.0646
  Val Loss: 0.0866
  Optimal Threshold: 0.440
  Balanced F1: 0.7549 ⭐
  Overall F1: 0.7333
  F1 Normal (0): 0.7778
  F1 Depression (1): 0.7333
  Precision: 0.6111
  Recall: 0.9167
  ⏳ No improvement (13/15)
----------------------------------------------------------------------


Epoch 24/50: 100%|██████████| 14/14 [00:01<00:00, 12.64it/s, loss=0.099] 



Epoch 24/50
  Train Loss: 0.0589
  Val Loss: 0.1109
  Optimal Threshold: 0.340
  Balanced F1: 0.7549 ⭐
  Overall F1: 0.7333
  F1 Normal (0): 0.7778
  F1 Depression (1): 0.7333
  Precision: 0.6111
  Recall: 0.9167
  ⏳ No improvement (14/15)
----------------------------------------------------------------------


Epoch 25/50: 100%|██████████| 14/14 [00:01<00:00, 12.69it/s, loss=0.095] 



Epoch 25/50
  Train Loss: 0.0578
  Val Loss: 0.0925
  Optimal Threshold: 0.400
  Balanced F1: 0.7549 ⭐
  Overall F1: 0.7333
  F1 Normal (0): 0.7778
  F1 Depression (1): 0.7333
  Precision: 0.6111
  Recall: 0.9167
  ⏳ No improvement (15/15)

⚠️  Early stopping!

📊 Test Set 평가

🎯 Test Set Results:
  Balanced F1: 0.6117 ⭐
  Overall F1: 0.5294
  F1 Normal (0): 0.7241
  F1 Depression (1): 0.5294
  Precision: 0.4500
  Recall: 0.6429
  Specificity: 0.6562

분류 보고서:
              precision    recall  f1-score   support

      Normal     0.8077    0.6562    0.7241        32
  Depression     0.4500    0.6429    0.5294        14

    accuracy                         0.6522        46
   macro avg     0.6288    0.6496    0.6268        46
weighted avg     0.6988    0.6522    0.6649        46


📊 Fusion Strategy 비교 결과

EARLY     : Balanced F1 = 0.7786, Threshold = 0.420
LATE      : Balanced F1 = 0.6667, Threshold = 0.430
ADAPTIVE  : Balanced F1 = 0.7708, Threshold = 0.430

✅ 모든 작업 완료!



In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import json

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
WAVLM_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_wavlm_only.pkl")
WAV2VEC_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_with_text.pkl")

# 모델 기본 설정
WAVLM_DIM = 768
WAV2VEC_DIM = 768
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_ENHANCED_DIM = 6

# 학습 기본 설정
BATCH_SIZE = 8
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# =============================================================================
# Enhanced TTR Feature Extraction
# =============================================================================
def extract_enhanced_ttr_features(text):
    """TTR 관련 강화된 특징 추출"""
    if not text or len(text.strip()) == 0:
        return {
            'ttr': 0.0, 'ttr_log': 0.0, 'repetition_rate': 0.0,
            'unique_word_ratio': 0.0, 'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    tokens = text.lower().split()
    if len(tokens) == 0:
        return {
            'ttr': 0.0, 'ttr_log': 0.0, 'repetition_rate': 0.0,
            'unique_word_ratio': 0.0, 'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    unique_tokens = set(tokens)
    ttr = len(unique_tokens) / len(tokens)
    ttr_log = len(unique_tokens) / math.log(len(tokens) + 1)
    
    word_counts = {}
    for token in tokens:
        word_counts[token] = word_counts.get(token, 0) + 1
    
    repeated_words = sum(1 for count in word_counts.values() if count > 1)
    repetition_rate = repeated_words / len(unique_tokens) if len(unique_tokens) > 0 else 0.0
    
    unique_words = sum(1 for count in word_counts.values() if count == 1)
    unique_word_ratio = unique_words / len(tokens)
    
    function_words = {
        'the', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'with', 'by', 'from', 'up', 'about', 'into', 'through', 'during',
        'he', 'she', 'it', 'we', 'they', 'him', 'her', 'us', 'them',
        'my', 'your', 'his', 'her', 'its', 'our', 'their',
        'am', 'is', 'are', 'were', 'be', 'been', 'being',
        'have', 'has', 'had', 'do', 'does', 'did',
        'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could',
        'this', 'that', 'these', 'those',
        'what', 'which', 'who', 'when', 'where', 'why', 'how'
    }
    
    content_words = [token for token in tokens if token not in function_words]
    lexical_density = len(content_words) / len(tokens) if len(tokens) > 0 else 0.0
    
    word_lengths = [len(token) for token in tokens]
    word_length_variance = np.var(word_lengths) if len(word_lengths) > 1 else 0.0
    
    return {
        'ttr': ttr, 'ttr_log': ttr_log, 'repetition_rate': repetition_rate,
        'unique_word_ratio': unique_word_ratio, 'lexical_density': lexical_density,
        'word_length_variance': word_length_variance
    }


# =============================================================================
# Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# WavLM-Centric Auxiliary Fusion Model
# =============================================================================
class WavLMCentricAuxiliaryFusion(nn.Module):
    """
    WavLM을 주 특징으로, Wav2Vec을 보조 특징으로 사용
    
    전략:
    1. WavLM은 그대로 유지 (강한 특징)
    2. Wav2Vec은 압축하여 보조 정보만 제공
    3. 선택적 융합 (Selective Fusion)
    """
    def __init__(
        self,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        dim_feedforward=512,
        dropout=0.3,
        aux_compression_ratio=4,  # Wav2Vec 압축 비율
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(WavLMCentricAuxiliaryFusion, self).__init__()
        
        self.d_model = d_model
        self.aux_compression_ratio = aux_compression_ratio
        
        # Q-type embedding
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # TTR projection
        self.ttr_projection = nn.Sequential(
            nn.Linear(TTR_ENHANCED_DIM, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32)
        )
        
        # === WavLM: Main Feature (그대로 유지) ===
        self.wavlm_projection = nn.Linear(WAVLM_DIM, d_model)
        
        # === Wav2Vec: Auxiliary Feature (압축) ===
        aux_dim = d_model // aux_compression_ratio  # 예: 256 / 4 = 64
        self.wav2vec_compression = nn.Sequential(
            nn.Linear(WAV2VEC_DIM, aux_dim * 2),
            nn.LayerNorm(aux_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(aux_dim * 2, aux_dim)
        )
        
        # === Selective Fusion Gate ===
        # WavLM 특징을 기반으로 Wav2Vec의 기여도 결정
        gate_input_dim = WAVLM_DIM + aux_dim
        self.fusion_gate = nn.Sequential(
            nn.Linear(gate_input_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()  # 0~1 사이의 가중치
        )
        
        # Auxiliary 정보 통합
        self.aux_integration = nn.Linear(aux_dim, d_model)
        
        # Context projection
        context_dim = q_type_embed_dim + 32
        self.context_projection = nn.Linear(context_dim, d_model // 4)
        
        # Final feature combination
        self.feature_combiner = nn.Sequential(
            nn.Linear(d_model + d_model + d_model // 4, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
        
        # For analysis
        self.last_gates = None
    
    def forward(self, batch_wavlm, batch_wav2vec, batch_ttr_enhanced, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_wavlm.device
        
        # Embeddings
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttr_projected = self.ttr_projection(batch_ttr_enhanced)
        
        # === Main Feature: WavLM (Full Representation) ===
        wavlm_main = self.wavlm_projection(batch_wavlm)  # [N, d_model]
        
        # === Auxiliary Feature: Wav2Vec (Compressed) ===
        wav2vec_aux = self.wav2vec_compression(batch_wav2vec)  # [N, aux_dim]
        
        # === Selective Fusion ===
        # WavLM의 정보를 보고 Wav2Vec을 얼마나 사용할지 결정
        gate_input = torch.cat([batch_wavlm, wav2vec_aux], dim=1)
        fusion_weight = self.fusion_gate(gate_input)  # [N, 1]
        
        # Wav2Vec 보조 정보 통합 (선택적으로)
        wav2vec_integrated = self.aux_integration(wav2vec_aux)  # [N, d_model]
        wav2vec_weighted = wav2vec_integrated * fusion_weight
        
        # Context features
        context_feat = self.context_projection(torch.cat([q_type_embs, ttr_projected], dim=1))
        
        # === Combine Features ===
        # WavLM (main) + Wav2Vec (weighted auxiliary) + Context
        combined = torch.cat([
            wavlm_main,
            wav2vec_weighted,
            context_feat
        ], dim=1)
        
        combined_features = self.feature_combiner(combined)  # [N, d_model]
        
        # Store gates for analysis
        self.last_gates = fusion_weight.detach()
        
        # Split by participants
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # Padding
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    self.d_model,
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, padded_sequences], dim=1)
        
        # Extend mask for CLS token
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional encoding
        x = self.pos_encoder(x)
        
        # Transformer encoding
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # CLS output
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        # Attention weights
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate (동일)
# =============================================================================
class MultimodalUtteranceDataset(Dataset):
    def __init__(self, wavlm_data, wav2vec_data):
        self.data = []
        common_pids = set(wavlm_data.keys()) & set(wav2vec_data.keys())
        
        for pid in common_pids:
            if wavlm_data[pid]['num_utterances'] != wav2vec_data[pid]['num_utterances']:
                continue
            
            self.data.append({
                'pid': pid,
                'label': wavlm_data[pid]['label'],
                'wavlm_utterances': wavlm_data[pid]['utterances'],
                'wav2vec_utterances': wav2vec_data[pid]['utterances'],
                'num_utterances': wavlm_data[pid]['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def multimodal_collate_fn(batch):
    batch_labels = []
    all_wavlm_utterances = []
    all_wav2vec_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_wavlm_utterances.extend(item['wavlm_utterances'])
        all_wav2vec_utterances.extend(item['wav2vec_utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    batch_wavlm = []
    batch_wav2vec = []
    batch_ttr_enhanced = []
    batch_q_type_ids = []
    
    for wavlm_utt, wav2vec_utt in zip(all_wavlm_utterances, all_wav2vec_utterances):
        batch_wavlm.append(wavlm_utt['wavlm'])
        batch_wav2vec.append(wav2vec_utt['wav2vec'])
        batch_q_type_ids.append(wavlm_utt['q_type_id'])
        
        text = wavlm_utt.get('text', '')
        ttr_features = extract_enhanced_ttr_features(text)
        
        ttr_vector = [
            ttr_features['ttr'], ttr_features['ttr_log'],
            ttr_features['repetition_rate'], ttr_features['unique_word_ratio'],
            ttr_features['lexical_density'], ttr_features['word_length_variance']
        ]
        batch_ttr_enhanced.append(ttr_vector)
    
    batch_wavlm = torch.FloatTensor(np.array(batch_wavlm))
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttr_enhanced = torch.FloatTensor(np.array(batch_ttr_enhanced))
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    return {
        'batch_wavlm': batch_wavlm,
        'batch_wav2vec': batch_wav2vec,
        'batch_ttr_enhanced': batch_ttr_enhanced,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }


# =============================================================================
# 데이터 로드
# =============================================================================
def load_and_split_multimodal_data():
    print(f"{'='*70}")
    print(f"📂 Multimodal 데이터 로드 중... (WavLM-Centric)")
    print(f"{'='*70}")
    
    with open(WAVLM_DATA_PATH, 'rb') as f:
        wavlm_dataset = pickle.load(f)
    
    with open(WAV2VEC_DATA_PATH, 'rb') as f:
        wav2vec_dataset = pickle.load(f)
    
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    common_pids = set(wavlm_dataset.keys()) & set(wav2vec_dataset.keys())
    
    train_pids = [pid for pid in meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist() 
                  if pid in common_pids]
    val_pids = [pid for pid in meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
                if pid in common_pids]
    test_pids = [pid for pid in meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
                 if pid in common_pids]
    
    train_labels = [wavlm_dataset[pid]['label'] for pid in train_pids]
    val_labels = [wavlm_dataset[pid]['label'] for pid in val_pids]
    test_labels = [wavlm_dataset[pid]['label'] for pid in test_pids]
    
    print(f"\n공통 참가자 수: {len(common_pids)}")
    print(f"메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    train_wavlm = {pid: wavlm_dataset[pid] for pid in train_pids}
    train_wav2vec = {pid: wav2vec_dataset[pid] for pid in train_pids}
    val_wavlm = {pid: wavlm_dataset[pid] for pid in val_pids}
    val_wav2vec = {pid: wav2vec_dataset[pid] for pid in val_pids}
    test_wavlm = {pid: wavlm_dataset[pid] for pid in test_pids}
    test_wav2vec = {pid: wav2vec_dataset[pid] for pid in test_pids}
    
    train_dataset = MultimodalUtteranceDataset(train_wavlm, train_wav2vec)
    val_dataset = MultimodalUtteranceDataset(val_wavlm, val_wav2vec)
    test_dataset = MultimodalUtteranceDataset(test_wavlm, test_wav2vec)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                             collate_fn=multimodal_collate_fn, num_workers=0,
                             pin_memory=True if torch.cuda.is_available() else False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=multimodal_collate_fn, num_workers=0,
                           pin_memory=True if torch.cuda.is_available() else False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=multimodal_collate_fn, num_workers=0,
                            pin_memory=True if torch.cuda.is_available() else False)
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 함수
# =============================================================================
def evaluate_multimodal(model, dataloader, criterion, threshold=0.5):
    model.eval()
    all_labels = []
    all_probs = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                batch_q_type_ids, batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader)
    all_preds = (np.array(all_probs) > threshold).astype(int)
    
    overall_f1 = f1_score(all_labels, all_preds, average='binary')
    f1_per_class = f1_score(all_labels, all_preds, average=None)
    f1_normal = f1_per_class[0]
    f1_depression = f1_per_class[1]
    
    if f1_normal > 0 and f1_depression > 0:
        balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
    else:
        balanced_f1 = 0.0
    
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    tn = sum((l == 0 and p == 0) for l, p in zip(all_labels, all_preds))
    fp = sum((l == 0 and p == 1) for l, p in zip(all_labels, all_preds))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return {
        'loss': avg_loss, 'overall_f1': overall_f1, 'balanced_f1': balanced_f1,
        'f1_normal': f1_normal, 'f1_depression': f1_depression,
        'precision': precision, 'recall': recall, 'specificity': specificity,
        'labels': all_labels, 'probs': all_probs, 'preds': all_preds
    }


def find_optimal_threshold(labels, probs, metric='balanced_f1'):
    best_score = 0.0
    best_threshold = 0.5
    best_metrics = {
        'threshold': 0.5, 'balanced_f1': 0.0, 'overall_f1': 0.0,
        'f1_normal': 0.0, 'f1_depression': 0.0,
        'precision': 0.0, 'recall': 0.0
    }
    
    for thresh in np.arange(0.1, 0.9, 0.01):
        preds = (np.array(probs) > thresh).astype(int)
        if len(np.unique(preds)) < 2:
            continue
        
        overall_f1 = f1_score(labels, preds, average='binary', zero_division=0)
        f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
        
        if len(f1_per_class) < 2:
            continue
            
        f1_normal = f1_per_class[0]
        f1_depression = f1_per_class[1]
        
        if f1_normal > 0 and f1_depression > 0:
            balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
        else:
            balanced_f1 = 0.0
        
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        score = balanced_f1 if metric == 'balanced_f1' else overall_f1
        
        if score > best_score:
            best_score = score
            best_threshold = thresh
            best_metrics = {
                'threshold': thresh, 'balanced_f1': balanced_f1,
                'overall_f1': overall_f1, 'f1_normal': f1_normal,
                'f1_depression': f1_depression,
                'precision': precision, 'recall': recall
            }
    
    return best_threshold, best_metrics


# =============================================================================
# Training
# =============================================================================
def train_wavlm_centric_model(aux_compression_ratio=4):
    """WavLM 중심 보조 융합 모델 학습"""
    print(f"\n{'='*70}")
    print(f"🚀 WavLM-Centric Auxiliary Fusion 학습")
    print(f"   Compression Ratio: {aux_compression_ratio}x")
    print(f"{'='*70}\n")
    
    train_loader, val_loader, test_loader = load_and_split_multimodal_data()
    
    model = WavLMCentricAuxiliaryFusion(
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        dim_feedforward=512,
        dropout=0.3,
        aux_compression_ratio=aux_compression_ratio
    ).to(DEVICE)
    
    criterion = FocalLoss(alpha=0.6, gamma=2.0, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-6)
    scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
    
    best_balanced_f1 = 0.0
    best_threshold = 0.5
    patience_counter = 0
    
    history = {
        'train_loss': [], 'val_loss': [], 'val_balanced_f1': [],
        'val_overall_f1': [], 'val_f1_normal': [], 'val_f1_depression': [],
        'val_precision': [], 'val_recall': []
    }
    
    for epoch in range(NUM_EPOCHS):
        # Training
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
        for batch in pbar:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                batch_q_type_ids, batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        val_results = evaluate_multimodal(model, val_loader, criterion, threshold=0.5)
        opt_threshold, threshold_metrics = find_optimal_threshold(
            val_results['labels'], val_results['probs'], metric='balanced_f1'
        )
        
        # History
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_results['loss'])
        history['val_balanced_f1'].append(threshold_metrics['balanced_f1'])
        history['val_overall_f1'].append(threshold_metrics['overall_f1'])
        history['val_f1_normal'].append(threshold_metrics['f1_normal'])
        history['val_f1_depression'].append(threshold_metrics['f1_depression'])
        history['val_precision'].append(threshold_metrics['precision'])
        history['val_recall'].append(threshold_metrics['recall'])
        
        scheduler.step()
        
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {val_results['loss']:.4f}")
        print(f"  Threshold: {opt_threshold:.3f}")
        print(f"  Balanced F1: {threshold_metrics['balanced_f1']:.4f} ⭐")
        print(f"  Depression F1: {threshold_metrics['f1_depression']:.4f}")
        print(f"  Precision: {threshold_metrics['precision']:.4f} | Recall: {threshold_metrics['recall']:.4f}")
        
        # Best model
        if threshold_metrics['balanced_f1'] > best_balanced_f1:
            best_balanced_f1 = threshold_metrics['balanced_f1']
            best_threshold = opt_threshold
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'balanced_f1': best_balanced_f1,
                'optimal_threshold': best_threshold,
                'history': history,
                'aux_compression_ratio': aux_compression_ratio
            }, os.path.join(BASE_PATH, 'best_wavlm_centric_aux_model.pt'))
            
            print(f"  ✅ Best model saved! (Balanced F1: {best_balanced_f1:.4f})")
        else:
            patience_counter += 1
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping!")
                break
        print("-" * 70)
    
    # Test
    print(f"\n{'='*70}")
    print(f"📊 Test Set 평가")
    print(f"{'='*70}")
    
    model_path = os.path.join(BASE_PATH, 'best_wavlm_centric_aux_model.pt')
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        best_threshold = checkpoint['optimal_threshold']
        
        test_results = evaluate_multimodal(model, test_loader, nn.BCEWithLogitsLoss(), threshold=best_threshold)
        
        print(f"\n🎯 Test Set Results:")
        print(f"  Balanced F1: {test_results['balanced_f1']:.4f} ⭐")
        print(f"  Overall F1: {test_results['overall_f1']:.4f}")
        print(f"  F1 Normal: {test_results['f1_normal']:.4f}")
        print(f"  F1 Depression: {test_results['f1_depression']:.4f}")
        print(f"  Precision: {test_results['precision']:.4f}")
        print(f"  Recall: {test_results['recall']:.4f}")
        
        print(f"\n{'='*70}")
        print("분류 보고서:")
        print(f"{'='*70}")
        print(classification_report(
            test_results['labels'], test_results['preds'],
            target_names=['Normal', 'Depression'], digits=4
        ))
        
        # Gate 분석
        print(f"\n{'='*70}")
        print("Fusion Gate 분석 (Wav2Vec 사용 비율):")
        print(f"{'='*70}")
        if hasattr(model, 'last_gates') and model.last_gates is not None:
            gates = model.last_gates.cpu().numpy().flatten()
            print(f"  평균: {gates.mean():.3f}")
            print(f"  표준편차: {gates.std():.3f}")
            print(f"  최소: {gates.min():.3f} | 최대: {gates.max():.3f}")
            print(f"  중앙값: {np.median(gates):.3f}")
    
    return model, history


# =============================================================================
# 메인
# =============================================================================
if __name__ == "__main__":
    print(f"\n{'='*70}")
    print(f"🤖 WavLM-Centric Auxiliary Fusion Model")
    print(f"{'='*70}\n")
    
    # 여러 compression ratio 테스트
    compression_ratios = [4, 8]  # 4x, 8x 압축
    
    for ratio in compression_ratios:
        print(f"\n{'='*70}")
        print(f"Testing Compression Ratio: {ratio}x")
        print(f"{'='*70}\n")
        
        model, history = train_wavlm_centric_model(aux_compression_ratio=ratio)
    
    print(f"\n{'='*70}")
    print(f"✅ 모든 작업 완료!")
    print(f"{'='*70}\n")


🤖 WavLM-Centric Auxiliary Fusion Model


Testing Compression Ratio: 4x


🚀 WavLM-Centric Auxiliary Fusion 학습
   Compression Ratio: 4x

📂 Multimodal 데이터 로드 중... (WavLM-Centric)

공통 참가자 수: 186
메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)


Epoch 1/50: 100%|██████████| 14/14 [00:01<00:00, 10.54it/s, loss=0.0826]



Epoch 1/50
  Train Loss: 0.0866 | Val Loss: 0.0895
  Threshold: 0.470
  Balanced F1: 0.1418 ⭐
  Depression F1: 0.4762
  Precision: 0.3333 | Recall: 0.8333
  ✅ Best model saved! (Balanced F1: 0.1418)
----------------------------------------------------------------------


Epoch 2/50: 100%|██████████| 14/14 [00:01<00:00, 11.02it/s, loss=0.035] 



Epoch 2/50
  Train Loss: 0.0792 | Val Loss: 0.1157
  Threshold: 0.380
  Balanced F1: 0.4925 ⭐
  Depression F1: 0.4286
  Precision: 0.3750 | Recall: 0.5000
  ✅ Best model saved! (Balanced F1: 0.4925)
----------------------------------------------------------------------


...


Epoch 29/50
  Train Loss: 0.0533 | Val Loss: 0.0894
  Threshold: 0.340
  Balanced F1: 0.7549 ⭐
  Depression F1: 0.7333
  Precision: 0.6111 | Recall: 0.9167
  ⏳ No improvement (13/15)
----------------------------------------------------------------------


Epoch 30/50: 100%|██████████| 14/14 [00:01<00:00, 11.91it/s, loss=0.0305]



Epoch 30/50
  Train Loss: 0.0507 | Val Loss: 0.1079
  Threshold: 0.400
  Balanced F1: 0.7423 ⭐
  Depression F1: 0.6923
  Precision: 0.6429 | Recall: 0.7500
  ⏳ No improvement (14/15)
----------------------------------------------------------------------


Epoch 31/50: 100%|██████████| 14/14 [00:01<00:00, 11.98it/s, loss=0.0159]



Epoch 31/50
  Train Loss: 0.0504 | Val Loss: 0.1288
  Threshold: 0.340
  Balanced F1: 0.7423 ⭐
  Depression F1: 0.6923
  Precision: 0.6429 | Recall: 0.7500
  ⏳ No improvement (15/15)

⚠️  Early stopping!

📊 Test Set 평가

🎯 Test Set Results:
  Balanced F1: 0.6682 ⭐
  Overall F1: 0.5806
  F1 Normal: 0.7869
  F1 Depression: 0.5806
  Precision: 0.5294
  Recall: 0.6429

분류 보고서:
              precision    recall  f1-score   support

      Normal     0.8276    0.7500    0.7869        32
  Depression     0.5294    0.6429    0.5806        14

    accuracy                         0.7174        46
   macro avg     0.6785    0.6964    0.6838        46
weighted avg     0.7368    0.7174    0.7241        46


Fusion Gate 분석 (Wav2Vec 사용 비율):
  평균: 0.438
  표준편차: 0.010
  최소: 0.387 | 최대: 0.471
  중앙값: 0.437

✅ 모든 작업 완료!



In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
import random
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import json
import optuna
from optuna.trial import TrialState

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
WAVLM_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_wavlm_only.pkl")
WAV2VEC_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_with_text.pkl")

# 모델 기본 설정
WAVLM_DIM = 768
WAV2VEC_DIM = 768
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_ENHANCED_DIM = 6

# 학습 기본 설정
BATCH_SIZE = 8
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True


# =============================================================================
# Enhanced TTR Feature Extraction (원본 그대로)
# =============================================================================
def extract_enhanced_ttr_features(text):
    if not text or len(text.strip()) == 0:
        return {
            'ttr': 0.0, 'ttr_log': 0.0, 'repetition_rate': 0.0,
            'unique_word_ratio': 0.0, 'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    tokens = text.lower().split()
    if len(tokens) == 0:
        return {
            'ttr': 0.0, 'ttr_log': 0.0, 'repetition_rate': 0.0,
            'unique_word_ratio': 0.0, 'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    unique_tokens = set(tokens)
    ttr = len(unique_tokens) / len(tokens)
    ttr_log = len(unique_tokens) / math.log(len(tokens) + 1)
    
    word_counts = {}
    for token in tokens:
        word_counts[token] = word_counts.get(token, 0) + 1
    
    repeated_words = sum(1 for count in word_counts.values() if count > 1)
    repetition_rate = repeated_words / len(unique_tokens) if len(unique_tokens) > 0 else 0.0
    
    unique_words = sum(1 for count in word_counts.values() if count == 1)
    unique_word_ratio = unique_words / len(tokens)
    
    function_words = {
        'the', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'with', 'by', 'from', 'up', 'about', 'into', 'through', 'during',
        'he', 'she', 'it', 'we', 'they', 'him', 'her', 'us', 'them',
        'my', 'your', 'his', 'her', 'its', 'our', 'their',
        'am', 'is', 'are', 'were', 'be', 'been', 'being',
        'have', 'has', 'had', 'do', 'does', 'did',
        'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could',
        'this', 'that', 'these', 'those',
        'what', 'which', 'who', 'when', 'where', 'why', 'how'
    }
    
    content_words = [token for token in tokens if token not in function_words]
    lexical_density = len(content_words) / len(tokens) if len(tokens) > 0 else 0.0
    
    word_lengths = [len(token) for token in tokens]
    word_length_variance = np.var(word_lengths) if len(word_lengths) > 1 else 0.0
    
    return {
        'ttr': ttr, 'ttr_log': ttr_log, 'repetition_rate': repetition_rate,
        'unique_word_ratio': unique_word_ratio, 'lexical_density': lexical_density,
        'word_length_variance': word_length_variance
    }


# =============================================================================
# Loss Functions (원본 그대로)
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# WavLM-Centric Auxiliary Fusion Model (원본 그대로)
# =============================================================================
class WavLMCentricAuxiliaryFusion(nn.Module):
    def __init__(
        self,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        dim_feedforward=512,
        dropout=0.3,
        aux_compression_ratio=4,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(WavLMCentricAuxiliaryFusion, self).__init__()
        
        self.d_model = d_model
        self.aux_compression_ratio = aux_compression_ratio
        
        # Q-type embedding
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # TTR projection
        self.ttr_projection = nn.Sequential(
            nn.Linear(TTR_ENHANCED_DIM, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32)
        )
        
        # === WavLM: Main Feature ===
        self.wavlm_projection = nn.Linear(WAVLM_DIM, d_model)
        
        # === Wav2Vec: Auxiliary Feature ===
        aux_dim = d_model // aux_compression_ratio
        self.wav2vec_compression = nn.Sequential(
            nn.Linear(WAV2VEC_DIM, aux_dim * 2),
            nn.LayerNorm(aux_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(aux_dim * 2, aux_dim)
        )
        
        # === Selective Fusion Gate ===
        gate_input_dim = WAVLM_DIM + aux_dim
        self.fusion_gate = nn.Sequential(
            nn.Linear(gate_input_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Auxiliary 정보 통합
        self.aux_integration = nn.Linear(aux_dim, d_model)
        
        # Context projection
        context_dim = q_type_embed_dim + 32
        self.context_projection = nn.Linear(context_dim, d_model // 4)
        
        # Final feature combination
        self.feature_combiner = nn.Sequential(
            nn.Linear(d_model + d_model + d_model // 4, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
        self.last_gates = None
    
    def forward(self, batch_wavlm, batch_wav2vec, batch_ttr_enhanced, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_wavlm.device
        
        # Embeddings
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttr_projected = self.ttr_projection(batch_ttr_enhanced)
        
        # === Main Feature: WavLM ===
        wavlm_main = self.wavlm_projection(batch_wavlm)
        
        # === Auxiliary Feature: Wav2Vec ===
        wav2vec_aux = self.wav2vec_compression(batch_wav2vec)
        
        # === Selective Fusion ===
        gate_input = torch.cat([batch_wavlm, wav2vec_aux], dim=1)
        fusion_weight = self.fusion_gate(gate_input)
        
        wav2vec_integrated = self.aux_integration(wav2vec_aux)
        wav2vec_weighted = wav2vec_integrated * fusion_weight
        
        # Context features
        context_feat = self.context_projection(torch.cat([q_type_embs, ttr_projected], dim=1))
        
        # === Combine Features ===
        combined = torch.cat([wavlm_main, wav2vec_weighted, context_feat], dim=1)
        combined_features = self.feature_combiner(combined)
        
        self.last_gates = fusion_weight.detach()
        
        # Split by participants
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # Padding
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(max_num_utterances - num_utts, self.d_model, device=device)
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, padded_sequences], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional encoding
        x = self.pos_encoder(x)
        
        # Transformer encoding
        encoded = self.transformer_encoder(x, src_key_padding_mask=extended_mask)
        
        # CLS output
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        # Attention weights
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate (원본 그대로)
# =============================================================================
class MultimodalUtteranceDataset(Dataset):
    def __init__(self, wavlm_data, wav2vec_data):
        self.data = []
        common_pids = set(wavlm_data.keys()) & set(wav2vec_data.keys())
        
        for pid in common_pids:
            if wavlm_data[pid]['num_utterances'] != wav2vec_data[pid]['num_utterances']:
                continue
            
            self.data.append({
                'pid': pid,
                'label': wavlm_data[pid]['label'],
                'wavlm_utterances': wavlm_data[pid]['utterances'],
                'wav2vec_utterances': wav2vec_data[pid]['utterances'],
                'num_utterances': wavlm_data[pid]['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


def multimodal_collate_fn(batch):
    batch_labels = []
    all_wavlm_utterances = []
    all_wav2vec_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_wavlm_utterances.extend(item['wavlm_utterances'])
        all_wav2vec_utterances.extend(item['wav2vec_utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    batch_wavlm = []
    batch_wav2vec = []
    batch_ttr_enhanced = []
    batch_q_type_ids = []
    
    for wavlm_utt, wav2vec_utt in zip(all_wavlm_utterances, all_wav2vec_utterances):
        batch_wavlm.append(wavlm_utt['wavlm'])
        batch_wav2vec.append(wav2vec_utt['wav2vec'])
        batch_q_type_ids.append(wavlm_utt['q_type_id'])
        
        text = wavlm_utt.get('text', '')
        ttr_features = extract_enhanced_ttr_features(text)
        
        ttr_vector = [
            ttr_features['ttr'], ttr_features['ttr_log'],
            ttr_features['repetition_rate'], ttr_features['unique_word_ratio'],
            ttr_features['lexical_density'], ttr_features['word_length_variance']
        ]
        batch_ttr_enhanced.append(ttr_vector)
    
    batch_wavlm = torch.FloatTensor(np.array(batch_wavlm))
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttr_enhanced = torch.FloatTensor(np.array(batch_ttr_enhanced))
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    return {
        'batch_wavlm': batch_wavlm,
        'batch_wav2vec': batch_wav2vec,
        'batch_ttr_enhanced': batch_ttr_enhanced,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }


# =============================================================================
# 데이터 로드 (캐싱)
# =============================================================================
_cached_data = None

def load_and_split_multimodal_data(verbose=True):
    global _cached_data
    
    if _cached_data is not None:
        return _cached_data
    
    if verbose:
        print(f"{'='*70}")
        print(f"📂 Multimodal 데이터 로드 중...")
        print(f"{'='*70}")
    
    with open(WAVLM_DATA_PATH, 'rb') as f:
        wavlm_dataset = pickle.load(f)
    
    with open(WAV2VEC_DATA_PATH, 'rb') as f:
        wav2vec_dataset = pickle.load(f)
    
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset2.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    common_pids = set(wavlm_dataset.keys()) & set(wav2vec_dataset.keys())
    
    train_pids = [pid for pid in meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist() 
                  if pid in common_pids]
    val_pids = [pid for pid in meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
                if pid in common_pids]
    test_pids = [pid for pid in meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
                 if pid in common_pids]
    
    if verbose:
        train_labels = [wavlm_dataset[pid]['label'] for pid in train_pids]
        val_labels = [wavlm_dataset[pid]['label'] for pid in val_pids]
        test_labels = [wavlm_dataset[pid]['label'] for pid in test_pids]
        
        print(f"\n공통 참가자 수: {len(common_pids)}")
        print(f"Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
        print(f"Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
        print(f"Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    train_wavlm = {pid: wavlm_dataset[pid] for pid in train_pids}
    train_wav2vec = {pid: wav2vec_dataset[pid] for pid in train_pids}
    val_wavlm = {pid: wavlm_dataset[pid] for pid in val_pids}
    val_wav2vec = {pid: wav2vec_dataset[pid] for pid in val_pids}
    test_wavlm = {pid: wavlm_dataset[pid] for pid in test_pids}
    test_wav2vec = {pid: wav2vec_dataset[pid] for pid in test_pids}
    
    train_dataset = MultimodalUtteranceDataset(train_wavlm, train_wav2vec)
    val_dataset = MultimodalUtteranceDataset(val_wavlm, val_wav2vec)
    test_dataset = MultimodalUtteranceDataset(test_wavlm, test_wav2vec)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                             collate_fn=multimodal_collate_fn, num_workers=0,
                             pin_memory=True if torch.cuda.is_available() else False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           collate_fn=multimodal_collate_fn, num_workers=0,
                           pin_memory=True if torch.cuda.is_available() else False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=multimodal_collate_fn, num_workers=0,
                            pin_memory=True if torch.cuda.is_available() else False)
    
    _cached_data = (train_loader, val_loader, test_loader)
    return _cached_data


# =============================================================================
# 평가 함수 (원본 그대로)
# =============================================================================
def evaluate_multimodal(model, dataloader, criterion, threshold=0.5):
    model.eval()
    all_labels = []
    all_probs = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                batch_q_type_ids, batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader)
    all_preds = (np.array(all_probs) > threshold).astype(int)
    
    if len(np.unique(all_preds)) < 2:
        return {
            'loss': avg_loss, 'overall_f1': 0.0, 'balanced_f1': 0.0,
            'f1_normal': 0.0, 'f1_depression': 0.0,
            'precision': 0.0, 'recall': 0.0, 'specificity': 0.0,
            'labels': all_labels, 'probs': all_probs, 'preds': all_preds
        }
    
    overall_f1 = f1_score(all_labels, all_preds, average='binary')
    f1_per_class = f1_score(all_labels, all_preds, average=None)
    f1_normal = f1_per_class[0]
    f1_depression = f1_per_class[1]
    
    if f1_normal > 0 and f1_depression > 0:
        balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
    else:
        balanced_f1 = 0.0
    
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    tn = sum((l == 0 and p == 0) for l, p in zip(all_labels, all_preds))
    fp = sum((l == 0 and p == 1) for l, p in zip(all_labels, all_preds))
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return {
        'loss': avg_loss, 'overall_f1': overall_f1, 'balanced_f1': balanced_f1,
        'f1_normal': f1_normal, 'f1_depression': f1_depression,
        'precision': precision, 'recall': recall, 'specificity': specificity,
        'labels': all_labels, 'probs': all_probs, 'preds': all_preds
    }


def find_optimal_threshold(labels, probs, metric='balanced_f1'):
    best_score = 0.0
    best_threshold = 0.5
    best_metrics = {
        'threshold': 0.5, 'balanced_f1': 0.0, 'overall_f1': 0.0,
        'f1_normal': 0.0, 'f1_depression': 0.0,
        'precision': 0.0, 'recall': 0.0
    }
    
    for thresh in np.arange(0.1, 0.9, 0.01):
        preds = (np.array(probs) > thresh).astype(int)
        if len(np.unique(preds)) < 2:
            continue
        
        overall_f1 = f1_score(labels, preds, average='binary', zero_division=0)
        f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
        
        if len(f1_per_class) < 2:
            continue
            
        f1_normal = f1_per_class[0]
        f1_depression = f1_per_class[1]
        
        if f1_normal > 0 and f1_depression > 0:
            balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
        else:
            balanced_f1 = 0.0
        
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        score = balanced_f1 if metric == 'balanced_f1' else overall_f1
        
        if score > best_score:
            best_score = score
            best_threshold = thresh
            best_metrics = {
                'threshold': thresh, 'balanced_f1': balanced_f1,
                'overall_f1': overall_f1, 'f1_normal': f1_normal,
                'f1_depression': f1_depression,
                'precision': precision, 'recall': recall
            }
    
    return best_threshold, best_metrics


# =============================================================================
# Depression-Focused Metric (핵심 변경)
# =============================================================================
def compute_depression_focused_score(f1_depression, f1_normal, min_normal_f1=0.65):
    """
    Depression F1을 우선하되, Normal F1이 최소 기준 이상일 때만 유효한 점수 반환
    
    예시:
    - Depression 0.58, Normal 0.70 → score = 0.58 (유효)
    - Depression 0.50, Normal 0.80 → score = 0.50 (유효하지만 낮음)
    - Depression 0.60, Normal 0.55 → score = 0.60 * 0.55/0.65 = 0.51 (패널티)
    
    이렇게 하면:
    - Normal 0.8, Depression 0.5 → 0.5
    - Normal 0.7, Depression 0.58 → 0.58 (이게 선택됨!)
    """
    if f1_normal >= min_normal_f1:
        # Normal F1이 기준 이상이면 Depression F1 그대로 사용
        return f1_depression
    else:
        # Normal F1이 기준 미만이면 패널티 적용
        penalty = f1_normal / min_normal_f1
        return f1_depression * penalty


def find_optimal_threshold_depression_focused(labels, probs, min_normal_f1=0.65):
    """
    Depression F1을 최대화하되, Normal F1이 min_normal_f1 이상인 threshold 탐색
    """
    best_score = 0.0
    best_threshold = 0.5
    best_metrics = {
        'threshold': 0.5, 'balanced_f1': 0.0, 'overall_f1': 0.0,
        'f1_normal': 0.0, 'f1_depression': 0.0,
        'precision': 0.0, 'recall': 0.0,
        'depression_focused_score': 0.0
    }
    
    for thresh in np.arange(0.1, 0.9, 0.01):
        preds = (np.array(probs) > thresh).astype(int)
        if len(np.unique(preds)) < 2:
            continue
        
        overall_f1 = f1_score(labels, preds, average='binary', zero_division=0)
        f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
        
        if len(f1_per_class) < 2:
            continue
            
        f1_normal = f1_per_class[0]
        f1_depression = f1_per_class[1]
        
        if f1_normal > 0 and f1_depression > 0:
            balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
        else:
            balanced_f1 = 0.0
        
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        # Depression-focused score 계산
        score = compute_depression_focused_score(f1_depression, f1_normal, min_normal_f1)
        
        if score > best_score:
            best_score = score
            best_threshold = thresh
            best_metrics = {
                'threshold': thresh, 'balanced_f1': balanced_f1,
                'overall_f1': overall_f1, 'f1_normal': f1_normal,
                'f1_depression': f1_depression,
                'precision': precision, 'recall': recall,
                'depression_focused_score': score
            }
    
    return best_threshold, best_metrics


# =============================================================================
# Optuna Objective Function (Depression-Focused)
# =============================================================================
def objective(trial):
    """Optuna objective function - Depression F1 최적화 (Normal F1 최소 기준 유지)"""
    
    set_seed(42)
    
    # ======================
    # 하이퍼파라미터 탐색 범위
    # ======================
    
    # Loss 관련
    focal_alpha = trial.suggest_float('focal_alpha', 0.6, 0.85)  # Depression에 더 집중
    focal_gamma = trial.suggest_float('focal_gamma', 1.5, 3.0)
    label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.15)
    
    # 모델 관련
    dropout = trial.suggest_float('dropout', 0.2, 0.4)
    aux_compression_ratio = trial.suggest_categorical('aux_compression_ratio', [2, 4, 8])
    
    # 학습 관련
    lr = trial.suggest_float('lr', 5e-5, 3e-4, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    warmup_epochs = trial.suggest_int('warmup_epochs', 3, 7)
    
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_multimodal_data(verbose=False)
    
    # 모델 생성
    model = WavLMCentricAuxiliaryFusion(
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        dim_feedforward=512,
        dropout=dropout,
        aux_compression_ratio=aux_compression_ratio
    ).to(DEVICE)
    
    # Loss & Optimizer
    criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma, label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Scheduler
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
    )
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-6
    )
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
    )
    
    best_score = 0.0
    patience_counter = 0
    
    for epoch in range(NUM_EPOCHS):
        # Training
        model.train()
        
        for batch in train_loader:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                batch_q_type_ids, batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
        
        scheduler.step()
        
        # Validation - Depression-focused threshold 탐색
        val_results = evaluate_multimodal(model, val_loader, criterion, threshold=0.5)
        opt_threshold, threshold_metrics = find_optimal_threshold_depression_focused(
            val_results['labels'], val_results['probs'], min_normal_f1=0.65
        )
        
        current_score = threshold_metrics['depression_focused_score']
        
        # Pruning
        trial.report(current_score, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
        # Early stopping check
        if current_score > best_score:
            best_score = current_score
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                break
    
    return best_score


# =============================================================================
# 최적 파라미터로 최종 학습 (Depression-Focused)
# =============================================================================
def train_with_best_params(best_params, num_runs=3):
    """최적 파라미터로 여러 번 학습하여 안정적인 결과 얻기 - Depression F1 우선"""
    
    print(f"\n{'='*70}")
    print(f"🎯 Best Parameters로 최종 학습 ({num_runs} runs)")
    print(f"   Metric: Depression F1 (with Normal F1 >= 0.65 constraint)")
    print(f"{'='*70}")
    print(f"Parameters: {best_params}")
    
    train_loader, val_loader, test_loader = load_and_split_multimodal_data(verbose=True)
    
    all_results = []
    best_model_state = None
    best_overall_f1 = 0.0
    
    for run in range(num_runs):
        print(f"\n--- Run {run+1}/{num_runs} ---")
        set_seed(42 + run * 100)
        
        # 모델 생성
        model = WavLMCentricAuxiliaryFusion(
            d_model=256,
            nhead=8,
            num_encoder_layers=3,
            dim_feedforward=512,
            dropout=best_params['dropout'],
            aux_compression_ratio=best_params['aux_compression_ratio']
        ).to(DEVICE)
        
        # Loss & Optimizer
        criterion = FocalLoss(
            alpha=best_params['focal_alpha'],
            gamma=best_params['focal_gamma'],
            label_smoothing=best_params['label_smoothing']
        )
        optimizer = optim.AdamW(
            model.parameters(),
            lr=best_params['lr'],
            weight_decay=best_params['weight_decay']
        )
        
        # Scheduler
        warmup_epochs = best_params['warmup_epochs']
        warmup_scheduler = optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
        )
        cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-6
        )
        scheduler = optim.lr_scheduler.SequentialLR(
            optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
        )
        
        best_score = 0.0
        best_threshold = 0.5
        patience_counter = 0
        run_best_state = None
        
        for epoch in range(NUM_EPOCHS):
            # Training
            model.train()
            train_loss = 0.0
            
            pbar = tqdm(train_loader, desc=f"Run {run+1} Epoch {epoch+1}")
            for batch in pbar:
                batch_wavlm = batch['batch_wavlm'].to(DEVICE)
                batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
                batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
                batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
                batch_labels = batch['batch_labels'].to(DEVICE)
                
                optimizer.zero_grad()
                logits, _ = model(
                    batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                    batch_q_type_ids, batch['num_utterances_list']
                )
                
                loss = criterion(logits, batch_labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
                optimizer.step()
                
                train_loss += loss.item()
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
            scheduler.step()
            
            # Validation - Depression-focused
            val_results = evaluate_multimodal(model, val_loader, criterion, threshold=0.5)
            opt_threshold, threshold_metrics = find_optimal_threshold_depression_focused(
                val_results['labels'], val_results['probs'], min_normal_f1=0.65
            )
            
            current_score = threshold_metrics['depression_focused_score']
            
            if current_score > best_score:
                best_score = current_score
                best_threshold = opt_threshold
                patience_counter = 0
                run_best_state = model.state_dict().copy()
                print(f"  Epoch {epoch+1}: Depression F1={threshold_metrics['f1_depression']:.4f}, "
                      f"Normal F1={threshold_metrics['f1_normal']:.4f} ✓")
            else:
                patience_counter += 1
                if patience_counter >= EARLY_STOPPING_PATIENCE:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break
        
        # Test evaluation for this run
        if run_best_state is not None:
            model.load_state_dict(run_best_state)
        
        test_results = evaluate_multimodal(model, test_loader, nn.BCEWithLogitsLoss(), threshold=best_threshold)
        test_opt_threshold, test_metrics = find_optimal_threshold_depression_focused(
            test_results['labels'], test_results['probs'], min_normal_f1=0.65
        )
        
        run_result = {
            'run': run + 1,
            'val_score': best_score,
            'val_threshold': best_threshold,
            'test_balanced_f1': test_metrics['balanced_f1'],
            'test_f1_depression': test_metrics['f1_depression'],
            'test_f1_normal': test_metrics['f1_normal'],
            'test_threshold': test_opt_threshold,
            'test_score': test_metrics['depression_focused_score']
        }
        all_results.append(run_result)
        
        print(f"\n  Run {run+1} Test Results:")
        print(f"    Depression F1: {test_metrics['f1_depression']:.4f} ⭐")
        print(f"    Normal F1: {test_metrics['f1_normal']:.4f}")
        print(f"    Balanced F1: {test_metrics['balanced_f1']:.4f}")
        
        # Track best overall
        if test_metrics['f1_depression'] > best_overall_f1:
            best_overall_f1 = test_metrics['f1_depression']
            best_model_state = run_best_state
    
    # Summary
    print(f"\n{'='*70}")
    print(f"📊 Summary of {num_runs} Runs")
    print(f"{'='*70}")
    
    avg_depression_f1 = np.mean([r['test_f1_depression'] for r in all_results])
    std_depression_f1 = np.std([r['test_f1_depression'] for r in all_results])
    avg_normal_f1 = np.mean([r['test_f1_normal'] for r in all_results])
    avg_balanced_f1 = np.mean([r['test_balanced_f1'] for r in all_results])
    
    print(f"  Avg Depression F1: {avg_depression_f1:.4f} ± {std_depression_f1:.4f} ⭐")
    print(f"  Avg Normal F1: {avg_normal_f1:.4f}")
    print(f"  Avg Balanced F1: {avg_balanced_f1:.4f}")
    print(f"  Best Depression F1: {best_overall_f1:.4f}")
    
    # Save best model
    if best_model_state is not None:
        torch.save({
            'model_state_dict': best_model_state,
            'best_params': best_params,
            'results': all_results
        }, os.path.join(BASE_PATH, 'optuna_best_model.pt'))
        print(f"\n  Best model saved!")
    
    return all_results, best_model_state


# =============================================================================
# 메인: Optuna 최적화 실행
# =============================================================================
def run_optuna_optimization(n_trials=50):
    """Optuna 하이퍼파라미터 최적화 실행 - Depression F1 최적화"""
    
    print(f"\n{'='*70}")
    print(f"🔍 Optuna Hyperparameter Optimization")
    print(f"   Trials: {n_trials}")
    print(f"   Target: Depression F1 (with Normal F1 >= 0.65)")
    print(f"{'='*70}\n")
    
    # 데이터 미리 로드
    load_and_split_multimodal_data(verbose=True)
    
    # Optuna study 생성
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )
    
    # 최적화 실행
    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True,
        callbacks=[lambda study, trial: print(f"Trial {trial.number}: Score={trial.value:.4f}")]
    )
    
    # 결과 출력
    print(f"\n{'='*70}")
    print(f"🏆 Optimization Results")
    print(f"{'='*70}")
    
    print(f"\nBest Trial:")
    print(f"  Value (Depression-Focused Score): {study.best_trial.value:.4f}")
    print(f"  Params:")
    for key, value in study.best_trial.params.items():
        print(f"    {key}: {value}")
    
    # Top 5 trials
    print(f"\nTop 5 Trials:")
    trials_df = study.trials_dataframe()
    trials_df = trials_df.sort_values('value', ascending=False).head(5)
    print(trials_df[['number', 'value', 'params_focal_alpha', 'params_focal_gamma', 
                     'params_dropout', 'params_lr']].to_string())
    
    # 최적 파라미터로 최종 학습
    best_params = study.best_trial.params
    results, best_model = train_with_best_params(best_params, num_runs=3)
    
    # 결과 저장
    final_results = {
        'best_params': best_params,
        'best_trial_value': study.best_trial.value,
        'final_results': results,
        'metric': 'depression_focused_score (Depression F1 with Normal F1 >= 0.65 constraint)'
    }
    
    with open(os.path.join(BASE_PATH, 'optuna_results.json'), 'w') as f:
        json.dump(final_results, f, indent=2)
    
    return study, results


# =============================================================================
# 메인
# =============================================================================
if __name__ == "__main__":
    print(f"\n{'='*70}")
    print(f"🤖 WavLM-Centric Model + Optuna Optimization")
    print(f"{'='*70}\n")
    
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    
    # Optuna 최적화 실행 (50 trials)
    study, results = run_optuna_optimization(n_trials=50)
    
    print(f"\n{'='*70}")
    print(f"✅ Optimization Complete!")
    print(f"{'='*70}\n")


🤖 WavLM-Centric Model + Optuna Optimization

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU

🔍 Optuna Hyperparameter Optimization
   Trials: 50
   Target: Depression F1 (with Normal F1 >= 0.65)

📂 Multimodal 데이터 로드 중...


[I 2025-12-13 17:33:04,935] A new study created in memory with name: no-name-95a98f98-11a1-421b-99e1-315c1cf4cbba



공통 참가자 수: 186
Train: 106 (0: 75, 1: 31)
Val:   33 (0: 21, 1: 12)
Test:  45 (0: 31, 1: 14)


  0%|          | 0/50 [00:00<?, ?it/s]

c:\Users\Lenovo\anaconda3\lib\site-packages\torch\optim\lr_scheduler.py:156: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


...

  Epoch 4: Depression F1=0.7407, Normal F1=0.8205 ✓


Run 3 Epoch 5: 100%|██████████| 14/14 [00:01<00:00, 11.97it/s, loss=0.0456]


  Epoch 5: Depression F1=0.7692, Normal F1=0.8500 ✓


Run 3 Epoch 6: 100%|██████████| 14/14 [00:01<00:00, 11.91it/s, loss=0.0604]
c:\Users\Lenovo\anaconda3\lib\site-packages\torch\optim\lr_scheduler.py:156: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)
Run 3 Epoch 20: 100%|██████████| 14/14 [00:01<00:00, 11.28it/s, loss=0.0115]


  Early stopping at epoch 20

  Run 3 Test Results:
    Depression F1: 0.5946 ⭐
    Normal F1: 0.7170
    Balanced F1: 0.6501

📊 Summary of 3 Runs
  Avg Depression F1: 0.6145 ± 0.0206 ⭐
  Avg Normal F1: 0.7759
  Avg Balanced F1: 0.6856
  Best Depression F1: 0.6429

  Best model saved!

✅ Optimization Complete!



In [ ]:
print(f"Average fusion gate weight: {model.last_gates.mean():.4f}")
print(f"Gate weight std: {model.last_gates.std():.4f}")

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import pandas as pd
import pickle
import os
import math
import random
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import json
from copy import deepcopy

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
WAVLM_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_wavlm_only.pkl")
WAV2VEC_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_with_text.pkl")

# 모델 기본 설정
WAVLM_DIM = 768
WAV2VEC_DIM = 768
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_ENHANCED_DIM = 6

# 학습 기본 설정
BATCH_SIZE = 8
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

# K-Fold 설정
N_FOLDS = 5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True


# =============================================================================
# Enhanced TTR Feature Extraction
# =============================================================================
def extract_enhanced_ttr_features(text):
    if not text or len(text.strip()) == 0:
        return {
            'ttr': 0.0, 'ttr_log': 0.0, 'repetition_rate': 0.0,
            'unique_word_ratio': 0.0, 'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    tokens = text.lower().split()
    if len(tokens) == 0:
        return {
            'ttr': 0.0, 'ttr_log': 0.0, 'repetition_rate': 0.0,
            'unique_word_ratio': 0.0, 'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    unique_tokens = set(tokens)
    ttr = len(unique_tokens) / len(tokens)
    ttr_log = len(unique_tokens) / math.log(len(tokens) + 1)
    
    word_counts = {}
    for token in tokens:
        word_counts[token] = word_counts.get(token, 0) + 1
    
    repeated_words = sum(1 for count in word_counts.values() if count > 1)
    repetition_rate = repeated_words / len(unique_tokens) if len(unique_tokens) > 0 else 0.0
    
    unique_words = sum(1 for count in word_counts.values() if count == 1)
    unique_word_ratio = unique_words / len(tokens)
    
    function_words = {
        'the', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'with', 'by', 'from', 'up', 'about', 'into', 'through', 'during',
        'he', 'she', 'it', 'we', 'they', 'him', 'her', 'us', 'them',
        'my', 'your', 'his', 'her', 'its', 'our', 'their',
        'am', 'is', 'are', 'were', 'be', 'been', 'being',
        'have', 'has', 'had', 'do', 'does', 'did',
        'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could',
        'this', 'that', 'these', 'those',
        'what', 'which', 'who', 'when', 'where', 'why', 'how'
    }
    
    content_words = [token for token in tokens if token not in function_words]
    lexical_density = len(content_words) / len(tokens) if len(tokens) > 0 else 0.0
    
    word_lengths = [len(token) for token in tokens]
    word_length_variance = np.var(word_lengths) if len(word_lengths) > 1 else 0.0
    
    return {
        'ttr': ttr, 'ttr_log': ttr_log, 'repetition_rate': repetition_rate,
        'unique_word_ratio': unique_word_ratio, 'lexical_density': lexical_density,
        'word_length_variance': word_length_variance
    }


# =============================================================================
# Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# WavLM-Centric Auxiliary Fusion Model
# =============================================================================
class WavLMCentricAuxiliaryFusion(nn.Module):
    def __init__(
        self,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        dim_feedforward=512,
        dropout=0.3,
        aux_compression_ratio=4,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(WavLMCentricAuxiliaryFusion, self).__init__()
        
        self.d_model = d_model
        self.aux_compression_ratio = aux_compression_ratio
        
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        self.ttr_projection = nn.Sequential(
            nn.Linear(TTR_ENHANCED_DIM, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32)
        )
        
        self.wavlm_projection = nn.Linear(WAVLM_DIM, d_model)
        
        aux_dim = d_model // aux_compression_ratio
        self.wav2vec_compression = nn.Sequential(
            nn.Linear(WAV2VEC_DIM, aux_dim * 2),
            nn.LayerNorm(aux_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(aux_dim * 2, aux_dim)
        )
        
        gate_input_dim = WAVLM_DIM + aux_dim
        self.fusion_gate = nn.Sequential(
            nn.Linear(gate_input_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        self.aux_integration = nn.Linear(aux_dim, d_model)
        
        context_dim = q_type_embed_dim + 32
        self.context_projection = nn.Linear(context_dim, d_model // 4)
        
        self.feature_combiner = nn.Sequential(
            nn.Linear(d_model + d_model + d_model // 4, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
        self.last_gates = None
    
    def forward(self, batch_wavlm, batch_wav2vec, batch_ttr_enhanced, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_wavlm.device
        
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttr_projected = self.ttr_projection(batch_ttr_enhanced)
        
        wavlm_main = self.wavlm_projection(batch_wavlm)
        wav2vec_aux = self.wav2vec_compression(batch_wav2vec)
        
        gate_input = torch.cat([batch_wavlm, wav2vec_aux], dim=1)
        fusion_weight = self.fusion_gate(gate_input)
        
        wav2vec_integrated = self.aux_integration(wav2vec_aux)
        wav2vec_weighted = wav2vec_integrated * fusion_weight
        
        context_feat = self.context_projection(torch.cat([q_type_embs, ttr_projected], dim=1))
        
        combined = torch.cat([wavlm_main, wav2vec_weighted, context_feat], dim=1)
        combined_features = self.feature_combiner(combined)
        
        self.last_gates = fusion_weight.detach()
        
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(max_num_utterances - num_utts, self.d_model, device=device)
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, padded_sequences], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        x = self.pos_encoder(x)
        
        encoded = self.transformer_encoder(x, src_key_padding_mask=extended_mask)
        
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate
# =============================================================================
class MultimodalUtteranceDataset(Dataset):
    def __init__(self, wavlm_data, wav2vec_data, pids=None):
        self.data = []
        
        if pids is None:
            common_pids = set(wavlm_data.keys()) & set(wav2vec_data.keys())
        else:
            common_pids = set(pids) & set(wavlm_data.keys()) & set(wav2vec_data.keys())
        
        for pid in common_pids:
            if wavlm_data[pid]['num_utterances'] != wav2vec_data[pid]['num_utterances']:
                continue
            
            self.data.append({
                'pid': pid,
                'label': wavlm_data[pid]['label'],
                'wavlm_utterances': wavlm_data[pid]['utterances'],
                'wav2vec_utterances': wav2vec_data[pid]['utterances'],
                'num_utterances': wavlm_data[pid]['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]
    
    def get_labels(self):
        return [item['label'] for item in self.data]
    
    def get_pids(self):
        return [item['pid'] for item in self.data]


def multimodal_collate_fn(batch):
    batch_labels = []
    all_wavlm_utterances = []
    all_wav2vec_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_wavlm_utterances.extend(item['wavlm_utterances'])
        all_wav2vec_utterances.extend(item['wav2vec_utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    batch_wavlm = []
    batch_wav2vec = []
    batch_ttr_enhanced = []
    batch_q_type_ids = []
    
    for wavlm_utt, wav2vec_utt in zip(all_wavlm_utterances, all_wav2vec_utterances):
        batch_wavlm.append(wavlm_utt['wavlm'])
        batch_wav2vec.append(wav2vec_utt['wav2vec'])
        batch_q_type_ids.append(wavlm_utt['q_type_id'])
        
        text = wavlm_utt.get('text', '')
        ttr_features = extract_enhanced_ttr_features(text)
        
        ttr_vector = [
            ttr_features['ttr'], ttr_features['ttr_log'],
            ttr_features['repetition_rate'], ttr_features['unique_word_ratio'],
            ttr_features['lexical_density'], ttr_features['word_length_variance']
        ]
        batch_ttr_enhanced.append(ttr_vector)
    
    batch_wavlm = torch.FloatTensor(np.array(batch_wavlm))
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttr_enhanced = torch.FloatTensor(np.array(batch_ttr_enhanced))
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    return {
        'batch_wavlm': batch_wavlm,
        'batch_wav2vec': batch_wav2vec,
        'batch_ttr_enhanced': batch_ttr_enhanced,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }


# =============================================================================
# 데이터 로드
# =============================================================================
def load_raw_data():
    """원시 데이터 로드"""
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중...")
    print(f"{'='*70}")
    
    with open(WAVLM_DATA_PATH, 'rb') as f:
        wavlm_dataset = pickle.load(f)
    
    with open(WAV2VEC_DATA_PATH, 'rb') as f:
        wav2vec_dataset = pickle.load(f)
    
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    return wavlm_dataset, wav2vec_dataset, meta_df


def get_data_splits(wavlm_dataset, wav2vec_dataset, meta_df):
    """Train+Val / Test 분할"""
    common_pids = set(wavlm_dataset.keys()) & set(wav2vec_dataset.keys())
    
    train_val_pids = [pid for pid in meta_df[meta_df['Group'].isin(['Train', 'Validation'])]['Participant_ID'].tolist() 
                      if pid in common_pids]
    test_pids = [pid for pid in meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
                 if pid in common_pids]
    
    train_val_labels = [wavlm_dataset[pid]['label'] for pid in train_val_pids]
    test_labels = [wavlm_dataset[pid]['label'] for pid in test_pids]
    
    print(f"\n데이터 분할:")
    print(f"  Train+Val: {len(train_val_pids)} (Normal: {train_val_labels.count(0)}, Depression: {train_val_labels.count(1)})")
    print(f"  Test: {len(test_pids)} (Normal: {test_labels.count(0)}, Depression: {test_labels.count(1)})")
    
    return train_val_pids, test_pids


# =============================================================================
# 평가 함수
# =============================================================================
def evaluate_model(model, dataloader, criterion, threshold=0.5):
    model.eval()
    all_labels = []
    all_probs = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                batch_q_type_ids, batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0
    
    return all_labels, all_probs, avg_loss


def compute_metrics(labels, probs, threshold=0.5):
    """메트릭 계산"""
    preds = (np.array(probs) > threshold).astype(int)
    
    if len(np.unique(preds)) < 2:
        return {
            'f1_depression': 0.0, 'f1_normal': 0.0, 'balanced_f1': 0.0,
            'precision': 0.0, 'recall': 0.0, 'depression_focused_score': 0.0
        }
    
    f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
    f1_normal = f1_per_class[0] if len(f1_per_class) > 0 else 0.0
    f1_depression = f1_per_class[1] if len(f1_per_class) > 1 else 0.0
    
    if f1_normal > 0 and f1_depression > 0:
        balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
    else:
        balanced_f1 = 0.0
    
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    
    # Depression-focused score
    min_normal_f1 = 0.65
    if f1_normal >= min_normal_f1:
        depression_focused_score = f1_depression
    else:
        penalty = f1_normal / min_normal_f1
        depression_focused_score = f1_depression * penalty
    
    return {
        'f1_depression': f1_depression,
        'f1_normal': f1_normal,
        'balanced_f1': balanced_f1,
        'precision': precision,
        'recall': recall,
        'depression_focused_score': depression_focused_score
    }


def find_optimal_threshold_depression_focused(labels, probs, min_normal_f1=0.65):
    """Depression F1을 최대화하는 threshold 탐색"""
    best_score = 0.0
    best_threshold = 0.5
    best_metrics = None
    
    for thresh in np.arange(0.1, 0.9, 0.01):
        metrics = compute_metrics(labels, probs, thresh)
        score = metrics['depression_focused_score']
        
        if score > best_score:
            best_score = score
            best_threshold = thresh
            best_metrics = metrics.copy()
            best_metrics['threshold'] = thresh
    
    if best_metrics is None:
        best_metrics = {'threshold': 0.5, 'f1_depression': 0.0, 'f1_normal': 0.0, 
                       'balanced_f1': 0.0, 'depression_focused_score': 0.0}
    
    return best_threshold, best_metrics


# =============================================================================
# Test-Time Augmentation (TTA)
# =============================================================================
def evaluate_with_tta(model, dataloader, n_augmentations=5, dropout_rate=0.1):
    """
    Test-Time Augmentation: Dropout을 활성화하여 여러 번 예측 후 평균
    """
    model.train()  # Dropout 활성화
    
    all_labels = []
    all_probs_list = [[] for _ in range(n_augmentations)]
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            if len(all_labels) == 0:
                all_labels.extend(batch_labels.cpu().numpy().flatten())
            elif len(all_labels) < len(batch_labels) + len(all_labels):
                all_labels.extend(batch_labels.cpu().numpy().flatten())
            
            for aug_idx in range(n_augmentations):
                logits, _ = model(
                    batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                    batch_q_type_ids, batch['num_utterances_list']
                )
                probs = torch.sigmoid(logits)
                all_probs_list[aug_idx].extend(probs.cpu().numpy().flatten())
    
    # 재수집 (labels)
    all_labels = []
    for batch in dataloader:
        all_labels.extend(batch['batch_labels'].numpy().flatten())
    
    # 확률 평균
    all_probs_array = np.array(all_probs_list)
    ensemble_probs = np.mean(all_probs_array, axis=0)
    
    model.eval()
    
    return all_labels, ensemble_probs


# =============================================================================
# K-Fold Cross-Validation 학습
# =============================================================================
def train_kfold_ensemble(wavlm_dataset, wav2vec_dataset, train_val_pids, test_pids, best_params, n_folds=5):
    """K-Fold Cross-Validation으로 앙상블 모델 학습"""
    
    print(f"\n{'='*70}")
    print(f"🔄 {n_folds}-Fold Cross-Validation Ensemble Training")
    print(f"{'='*70}")
    print(f"Parameters: {best_params}")
    
    # Train+Val 데이터셋 생성
    full_dataset = MultimodalUtteranceDataset(wavlm_dataset, wav2vec_dataset, pids=train_val_pids)
    labels = full_dataset.get_labels()
    
    # Test 데이터셋
    test_dataset = MultimodalUtteranceDataset(wavlm_dataset, wav2vec_dataset, pids=test_pids)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=multimodal_collate_fn, num_workers=0)
    
    # Stratified K-Fold
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_models = []
    fold_thresholds = []
    fold_val_results = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(range(len(full_dataset)), labels)):
        print(f"\n{'='*50}")
        print(f"📊 Fold {fold+1}/{n_folds}")
        print(f"{'='*50}")
        
        set_seed(42 + fold * 100)
        
        # Subset 생성
        train_subset = Subset(full_dataset, train_idx)
        val_subset = Subset(full_dataset, val_idx)
        
        train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                                 collate_fn=multimodal_collate_fn, num_workers=0)
        val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                               collate_fn=multimodal_collate_fn, num_workers=0)
        
        train_labels = [labels[i] for i in train_idx]
        val_labels = [labels[i] for i in val_idx]
        print(f"  Train: {len(train_idx)} (Dep: {sum(train_labels)})")
        print(f"  Val: {len(val_idx)} (Dep: {sum(val_labels)})")
        
        # 모델 생성
        model = WavLMCentricAuxiliaryFusion(
            d_model=256,
            nhead=8,
            num_encoder_layers=3,
            dim_feedforward=512,
            dropout=best_params.get('dropout', 0.3),
            aux_compression_ratio=best_params.get('aux_compression_ratio', 4)
        ).to(DEVICE)
        
        criterion = FocalLoss(
            alpha=best_params.get('focal_alpha', 0.7),
            gamma=best_params.get('focal_gamma', 2.0),
            label_smoothing=best_params.get('label_smoothing', 0.1)
        )
        
        optimizer = optim.AdamW(
            model.parameters(),
            lr=best_params.get('lr', 1e-4),
            weight_decay=best_params.get('weight_decay', 1e-4)
        )
        
        warmup_epochs = best_params.get('warmup_epochs', 5)
        warmup_scheduler = optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
        )
        cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-6
        )
        scheduler = optim.lr_scheduler.SequentialLR(
            optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
        )
        
        best_score = 0.0
        best_threshold = 0.5
        patience_counter = 0
        best_state = None
        
        for epoch in range(NUM_EPOCHS):
            # Training
            model.train()
            
            pbar = tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}")
            for batch in pbar:
                batch_wavlm = batch['batch_wavlm'].to(DEVICE)
                batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
                batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
                batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
                batch_labels_tensor = batch['batch_labels'].to(DEVICE)
                
                optimizer.zero_grad()
                logits, _ = model(
                    batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                    batch_q_type_ids, batch['num_utterances_list']
                )
                
                loss = criterion(logits, batch_labels_tensor)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
                optimizer.step()
                
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
            scheduler.step()
            
            # Validation
            val_labels_eval, val_probs, _ = evaluate_model(model, val_loader, criterion)
            opt_threshold, val_metrics = find_optimal_threshold_depression_focused(val_labels_eval, val_probs)
            
            current_score = val_metrics['depression_focused_score']
            
            if current_score > best_score:
                best_score = current_score
                best_threshold = opt_threshold
                patience_counter = 0
                best_state = deepcopy(model.state_dict())
                print(f"    Epoch {epoch+1}: Dep F1={val_metrics['f1_depression']:.4f}, "
                      f"Normal F1={val_metrics['f1_normal']:.4f} ✓")
            else:
                patience_counter += 1
                if patience_counter >= EARLY_STOPPING_PATIENCE:
                    print(f"    Early stopping at epoch {epoch+1}")
                    break
        
        # Best model 로드
        if best_state is not None:
            model.load_state_dict(best_state)
        
        fold_models.append(model)
        fold_thresholds.append(best_threshold)
        fold_val_results.append({
            'fold': fold + 1,
            'val_depression_f1': val_metrics['f1_depression'],
            'val_normal_f1': val_metrics['f1_normal'],
            'threshold': best_threshold
        })
        
        print(f"\n  Fold {fold+1} Best: Dep F1={val_metrics['f1_depression']:.4f}")
    
    return fold_models, fold_thresholds, fold_val_results, test_loader


# =============================================================================
# 앙상블 평가
# =============================================================================
def evaluate_ensemble(fold_models, test_loader, method='average'):
    """여러 fold 모델의 앙상블 평가"""
    
    print(f"\n{'='*70}")
    print(f"🎯 Ensemble Evaluation (Method: {method})")
    print(f"{'='*70}")
    
    all_labels = []
    all_probs_list = []
    
    # 각 모델의 예측 수집
    for fold_idx, model in enumerate(fold_models):
        model.eval()
        fold_probs = []
        
        with torch.no_grad():
            for batch in test_loader:
                batch_wavlm = batch['batch_wavlm'].to(DEVICE)
                batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
                batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
                batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
                
                if fold_idx == 0:
                    all_labels.extend(batch['batch_labels'].numpy().flatten())
                
                logits, _ = model(
                    batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                    batch_q_type_ids, batch['num_utterances_list']
                )
                probs = torch.sigmoid(logits)
                fold_probs.extend(probs.cpu().numpy().flatten())
        
        all_probs_list.append(fold_probs)
    
    # 앙상블 방법
    all_probs_array = np.array(all_probs_list)
    
    if method == 'average':
        ensemble_probs = np.mean(all_probs_array, axis=0)
    elif method == 'median':
        ensemble_probs = np.median(all_probs_array, axis=0)
    elif method == 'max':
        ensemble_probs = np.max(all_probs_array, axis=0)
    else:
        ensemble_probs = np.mean(all_probs_array, axis=0)
    
    return all_labels, ensemble_probs, all_probs_list


def evaluate_ensemble_with_tta(fold_models, test_loader, n_tta=5):
    """앙상블 + TTA"""
    
    print(f"\n{'='*70}")
    print(f"🎯 Ensemble + TTA Evaluation (TTA augmentations: {n_tta})")
    print(f"{'='*70}")
    
    all_labels = []
    all_probs_list = []
    
    for fold_idx, model in enumerate(fold_models):
        # TTA로 평가
        labels, tta_probs = evaluate_with_tta(model, test_loader, n_augmentations=n_tta)
        
        if fold_idx == 0:
            all_labels = labels
        
        all_probs_list.append(tta_probs)
    
    # 앙상블 (평균)
    ensemble_probs = np.mean(all_probs_list, axis=0)
    
    return all_labels, ensemble_probs


# =============================================================================
# 메인 실행
# =============================================================================
def run_kfold_ensemble_training(best_params=None):
    """K-Fold 앙상블 학습 및 평가 실행"""
    
    # 기본 파라미터 (Optuna에서 찾은 값 사용 가능)
    if best_params is None:
        best_params = {
            'focal_alpha': 0.7,
            'focal_gamma': 2.0,
            'label_smoothing': 0.1,
            'dropout': 0.3,
            'aux_compression_ratio': 4,
            'lr': 1e-4,
            'weight_decay': 1e-4,
            'warmup_epochs': 5
        }
    
    # 데이터 로드
    wavlm_dataset, wav2vec_dataset, meta_df = load_raw_data()
    train_val_pids, test_pids = get_data_splits(wavlm_dataset, wav2vec_dataset, meta_df)
    
    # K-Fold 학습
    fold_models, fold_thresholds, fold_val_results, test_loader = train_kfold_ensemble(
        wavlm_dataset, wav2vec_dataset, train_val_pids, test_pids, 
        best_params, n_folds=N_FOLDS
    )
    
    # ==========================================
    # 평가 방법 1: 단순 앙상블 (Average)
    # ==========================================
    labels, ensemble_probs_avg, individual_probs = evaluate_ensemble(fold_models, test_loader, method='average')
    opt_thresh_avg, metrics_avg = find_optimal_threshold_depression_focused(labels, ensemble_probs_avg)
    
    print(f"\n📊 Results - Ensemble (Average):")
    print(f"  Threshold: {opt_thresh_avg:.3f}")
    print(f"  Depression F1: {metrics_avg['f1_depression']:.4f} ⭐")
    print(f"  Normal F1: {metrics_avg['f1_normal']:.4f}")
    print(f"  Balanced F1: {metrics_avg['balanced_f1']:.4f}")
    
    # ==========================================
    # 평가 방법 2: 앙상블 (Median)
    # ==========================================
    _, ensemble_probs_median, _ = evaluate_ensemble(fold_models, test_loader, method='median')
    opt_thresh_med, metrics_med = find_optimal_threshold_depression_focused(labels, ensemble_probs_median)
    
    print(f"\n📊 Results - Ensemble (Median):")
    print(f"  Threshold: {opt_thresh_med:.3f}")
    print(f"  Depression F1: {metrics_med['f1_depression']:.4f} ⭐")
    print(f"  Normal F1: {metrics_med['f1_normal']:.4f}")
    print(f"  Balanced F1: {metrics_med['balanced_f1']:.4f}")
    
    # ==========================================
    # 평가 방법 3: 앙상블 + TTA
    # ==========================================
    labels_tta, ensemble_probs_tta = evaluate_ensemble_with_tta(fold_models, test_loader, n_tta=5)
    opt_thresh_tta, metrics_tta = find_optimal_threshold_depression_focused(labels_tta, ensemble_probs_tta)
    
    print(f"\n📊 Results - Ensemble + TTA:")
    print(f"  Threshold: {opt_thresh_tta:.3f}")
    print(f"  Depression F1: {metrics_tta['f1_depression']:.4f} ⭐")
    print(f"  Normal F1: {metrics_tta['f1_normal']:.4f}")
    print(f"  Balanced F1: {metrics_tta['balanced_f1']:.4f}")
    
    # ==========================================
    # 개별 Fold 결과
    # ==========================================
    print(f"\n📊 Individual Fold Results on Test Set:")
    for fold_idx, fold_probs in enumerate(individual_probs):
        opt_thresh, metrics = find_optimal_threshold_depression_focused(labels, fold_probs)
        print(f"  Fold {fold_idx+1}: Dep F1={metrics['f1_depression']:.4f}, "
              f"Normal F1={metrics['f1_normal']:.4f}, Thresh={opt_thresh:.3f}")
    
    # ==========================================
    # 최종 결과 선택
    # ==========================================
    all_results = {
        'average': {'metrics': metrics_avg, 'threshold': opt_thresh_avg, 'probs': ensemble_probs_avg},
        'median': {'metrics': metrics_med, 'threshold': opt_thresh_med, 'probs': ensemble_probs_median},
        'tta': {'metrics': metrics_tta, 'threshold': opt_thresh_tta, 'probs': ensemble_probs_tta}
    }
    
    # 가장 좋은 방법 선택
    best_method = max(all_results.keys(), key=lambda x: all_results[x]['metrics']['f1_depression'])
    best_result = all_results[best_method]
    
    print(f"\n{'='*70}")
    print(f"🏆 Best Method: {best_method.upper()}")
    print(f"{'='*70}")
    print(f"  Depression F1: {best_result['metrics']['f1_depression']:.4f} ⭐")
    print(f"  Normal F1: {best_result['metrics']['f1_normal']:.4f}")
    print(f"  Balanced F1: {best_result['metrics']['balanced_f1']:.4f}")
    print(f"  Optimal Threshold: {best_result['threshold']:.3f}")
    
    # Classification Report
    best_preds = (best_result['probs'] > best_result['threshold']).astype(int)
    print(f"\n분류 보고서:")
    print(classification_report(labels, best_preds, target_names=['Normal', 'Depression'], digits=4))
    
    # Confusion Matrix
    cm = confusion_matrix(labels, best_preds)
    print(f"Confusion Matrix:")
    print(f"  TN={cm[0,0]:3d}  FP={cm[0,1]:3d}")
    print(f"  FN={cm[1,0]:3d}  TP={cm[1,1]:3d}")
    
    # 모델 저장
    save_path = os.path.join(BASE_PATH, 'kfold_ensemble_models.pt')
    torch.save({
        'fold_models': [m.state_dict() for m in fold_models],
        'fold_thresholds': fold_thresholds,
        'fold_val_results': fold_val_results,
        'best_params': best_params,
        'best_method': best_method,
        'test_results': {
            'average': metrics_avg,
            'median': metrics_med,
            'tta': metrics_tta
        }
    }, save_path)
    print(f"\n✅ Models saved to: {save_path}")
    
    # JSON 결과 저장
    json_results = {
        'best_params': best_params,
        'fold_val_results': fold_val_results,
        'test_results': {
            'average': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                       for k, v in metrics_avg.items()},
            'median': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                      for k, v in metrics_med.items()},
            'tta': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                   for k, v in metrics_tta.items()}
        },
        'best_method': best_method
    }
    
    json_path = os.path.join(BASE_PATH, 'kfold_ensemble_results.json')
    with open(json_path, 'w') as f:
        json.dump(json_results, f, indent=2)
    print(f"✅ Results saved to: {json_path}")
    
    return fold_models, all_results


# =============================================================================
# 메인
# =============================================================================
if __name__ == "__main__":
    print(f"\n{'='*70}")
    print(f"🤖 K-Fold Ensemble + TTA Training")
    print(f"   Method: {N_FOLDS}-Fold CV + Test-Time Augmentation")
    print(f"{'='*70}\n")
    
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    
    # Optuna에서 찾은 최적 파라미터가 있으면 여기에 입력
    # 없으면 None으로 기본값 사용
    best_params = {
        'focal_alpha': 0.7,
        'focal_gamma': 2.0,
        'label_smoothing': 0.1,
        'dropout': 0.3,
        'aux_compression_ratio': 4,
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'warmup_epochs': 5
    }
    
    # 실행
    fold_models, results = run_kfold_ensemble_training(best_params)
    
    print(f"\n{'='*70}")
    print(f"✅ Training Complete!")
    print(f"{'='*70}\n")


🤖 K-Fold Ensemble + TTA Training
   Method: 5-Fold CV + Test-Time Augmentation

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
📂 데이터 로드 중...

데이터 분할:
  Train+Val: 140 (Normal: 97, Depression: 43)
  Test: 46 (Normal: 32, Depression: 14)

🔄 5-Fold Cross-Validation Ensemble Training
Parameters: {'focal_alpha': 0.7, 'focal_gamma': 2.0, 'label_smoothing': 0.1, 'dropout': 0.3, 'aux_compression_ratio': 4, 'lr': 0.0001, 'weight_decay': 0.0001, 'warmup_epochs': 5}

📊 Fold 1/5
  Train: 112 (Dep: 35)
  Val: 28 (Dep: 8)


Fold 1 Epoch 1: 100%|██████████| 14/14 [00:01<00:00, 13.04it/s, loss=0.0850]


    Epoch 1: Dep F1=0.4615, Normal F1=0.8372 ✓


Fold 1 Epoch 2: 100%|██████████| 14/14 [00:01<00:00, 13.20it/s, loss=0.1057]


    Epoch 2: Dep F1=0.5000, Normal F1=0.8636 ✓


...

    Epoch 17: Dep F1=0.7273, Normal F1=0.8235 ✓


Fold 5 Epoch 21: 100%|██████████| 14/14 [00:01<00:00, 12.74it/s, loss=0.0334]


    Epoch 21: Dep F1=0.7368, Normal F1=0.8649 ✓


Fold 5 Epoch 36: 100%|██████████| 14/14 [00:01<00:00, 10.18it/s, loss=0.0844]


    Early stopping at epoch 36

  Fold 5 Best: Dep F1=0.7000

🎯 Ensemble Evaluation (Method: average)

📊 Results - Ensemble (Average):
  Threshold: 0.550
  Depression F1: 0.6061 ⭐
  Normal F1: 0.7797
  Balanced F1: 0.6820

🎯 Ensemble Evaluation (Method: median)

📊 Results - Ensemble (Median):
  Threshold: 0.510
  Depression F1: 0.5946 ⭐
  Normal F1: 0.7273
  Balanced F1: 0.6543

🎯 Ensemble + TTA Evaluation (TTA augmentations: 5)

📊 Results - Ensemble + TTA:
  Threshold: 0.480
  Depression F1: 0.6061 ⭐
  Normal F1: 0.7797
  Balanced F1: 0.6820

📊 Individual Fold Results on Test Set:
  Fold 1: Dep F1=0.6111, Normal F1=0.7500, Thresh=0.510
  Fold 2: Dep F1=0.5714, Normal F1=0.8125, Thresh=0.550
  Fold 3: Dep F1=0.6250, Normal F1=0.8000, Thresh=0.710
  Fold 4: Dep F1=0.5882, Normal F1=0.7586, Thresh=0.550
  Fold 5: Dep F1=0.6111, Normal F1=0.7500, Thresh=0.540

🏆 Best Method: AVERAGE
  Depression F1: 0.6061 ⭐
  Normal F1: 0.7797
  Balanced F1: 0.6820
  Optimal Threshold: 0.550

분류 보고서:
   

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import pandas as pd
import pickle
import os
import math
import random
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import json
from copy import deepcopy

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
WAVLM_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_wavlm_only.pkl")
WAV2VEC_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset_with_text.pkl")

# 모델 기본 설정
WAVLM_DIM = 768
WAV2VEC_DIM = 768
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_ENHANCED_DIM = 6

# 학습 기본 설정
BATCH_SIZE = 8
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 15
GRADIENT_CLIP = 1.0

# K-Fold 설정
N_FOLDS = 5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True


# =============================================================================
# Enhanced TTR Feature Extraction
# =============================================================================
def extract_enhanced_ttr_features(text):
    if not text or len(text.strip()) == 0:
        return {
            'ttr': 0.0, 'ttr_log': 0.0, 'repetition_rate': 0.0,
            'unique_word_ratio': 0.0, 'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    tokens = text.lower().split()
    if len(tokens) == 0:
        return {
            'ttr': 0.0, 'ttr_log': 0.0, 'repetition_rate': 0.0,
            'unique_word_ratio': 0.0, 'lexical_density': 0.0,
            'word_length_variance': 0.0
        }
    
    unique_tokens = set(tokens)
    ttr = len(unique_tokens) / len(tokens)
    ttr_log = len(unique_tokens) / math.log(len(tokens) + 1)
    
    word_counts = {}
    for token in tokens:
        word_counts[token] = word_counts.get(token, 0) + 1
    
    repeated_words = sum(1 for count in word_counts.values() if count > 1)
    repetition_rate = repeated_words / len(unique_tokens) if len(unique_tokens) > 0 else 0.0
    
    unique_words = sum(1 for count in word_counts.values() if count == 1)
    unique_word_ratio = unique_words / len(tokens)
    
    function_words = {
        'the', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'with', 'by', 'from', 'up', 'about', 'into', 'through', 'during',
        'he', 'she', 'it', 'we', 'they', 'him', 'her', 'us', 'them',
        'my', 'your', 'his', 'her', 'its', 'our', 'their',
        'am', 'is', 'are', 'were', 'be', 'been', 'being',
        'have', 'has', 'had', 'do', 'does', 'did',
        'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could',
        'this', 'that', 'these', 'those',
        'what', 'which', 'who', 'when', 'where', 'why', 'how'
    }
    
    content_words = [token for token in tokens if token not in function_words]
    lexical_density = len(content_words) / len(tokens) if len(tokens) > 0 else 0.0
    
    word_lengths = [len(token) for token in tokens]
    word_length_variance = np.var(word_lengths) if len(word_lengths) > 1 else 0.0
    
    return {
        'ttr': ttr, 'ttr_log': ttr_log, 'repetition_rate': repetition_rate,
        'unique_word_ratio': unique_word_ratio, 'lexical_density': lexical_density,
        'word_length_variance': word_length_variance
    }


# =============================================================================
# Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# WavLM-Centric Auxiliary Fusion Model
# =============================================================================
class WavLMCentricAuxiliaryFusion(nn.Module):
    def __init__(
        self,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        dim_feedforward=512,
        dropout=0.3,
        aux_compression_ratio=4,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(WavLMCentricAuxiliaryFusion, self).__init__()
        
        self.d_model = d_model
        self.aux_compression_ratio = aux_compression_ratio
        
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        self.ttr_projection = nn.Sequential(
            nn.Linear(TTR_ENHANCED_DIM, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32)
        )
        
        self.wavlm_projection = nn.Linear(WAVLM_DIM, d_model)
        
        aux_dim = d_model // aux_compression_ratio
        self.wav2vec_compression = nn.Sequential(
            nn.Linear(WAV2VEC_DIM, aux_dim * 2),
            nn.LayerNorm(aux_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(aux_dim * 2, aux_dim)
        )
        
        gate_input_dim = WAVLM_DIM + aux_dim
        self.fusion_gate = nn.Sequential(
            nn.Linear(gate_input_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        self.aux_integration = nn.Linear(aux_dim, d_model)
        
        context_dim = q_type_embed_dim + 32
        self.context_projection = nn.Linear(context_dim, d_model // 4)
        
        self.feature_combiner = nn.Sequential(
            nn.Linear(d_model + d_model + d_model // 4, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
        self.last_gates = None
    
    def forward(self, batch_wavlm, batch_wav2vec, batch_ttr_enhanced, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_wavlm.device
        
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttr_projected = self.ttr_projection(batch_ttr_enhanced)
        
        wavlm_main = self.wavlm_projection(batch_wavlm)
        wav2vec_aux = self.wav2vec_compression(batch_wav2vec)
        
        gate_input = torch.cat([batch_wavlm, wav2vec_aux], dim=1)
        fusion_weight = self.fusion_gate(gate_input)
        
        wav2vec_integrated = self.aux_integration(wav2vec_aux)
        wav2vec_weighted = wav2vec_integrated * fusion_weight
        
        context_feat = self.context_projection(torch.cat([q_type_embs, ttr_projected], dim=1))
        
        combined = torch.cat([wavlm_main, wav2vec_weighted, context_feat], dim=1)
        combined_features = self.feature_combiner(combined)
        
        self.last_gates = fusion_weight.detach()
        
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(max_num_utterances - num_utts, self.d_model, device=device)
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, padded_sequences], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        x = self.pos_encoder(x)
        
        encoded = self.transformer_encoder(x, src_key_padding_mask=extended_mask)
        
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Dataset & Collate
# =============================================================================
class MultimodalUtteranceDataset(Dataset):
    def __init__(self, wavlm_data, wav2vec_data, pids=None):
        self.data = []
        
        if pids is None:
            common_pids = set(wavlm_data.keys()) & set(wav2vec_data.keys())
        else:
            common_pids = set(pids) & set(wavlm_data.keys()) & set(wav2vec_data.keys())
        
        for pid in common_pids:
            if wavlm_data[pid]['num_utterances'] != wav2vec_data[pid]['num_utterances']:
                continue
            
            self.data.append({
                'pid': pid,
                'label': wavlm_data[pid]['label'],
                'wavlm_utterances': wavlm_data[pid]['utterances'],
                'wav2vec_utterances': wav2vec_data[pid]['utterances'],
                'num_utterances': wavlm_data[pid]['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]
    
    def get_labels(self):
        return [item['label'] for item in self.data]
    
    def get_pids(self):
        return [item['pid'] for item in self.data]


def multimodal_collate_fn(batch):
    batch_labels = []
    all_wavlm_utterances = []
    all_wav2vec_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_wavlm_utterances.extend(item['wavlm_utterances'])
        all_wav2vec_utterances.extend(item['wav2vec_utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    batch_wavlm = []
    batch_wav2vec = []
    batch_ttr_enhanced = []
    batch_q_type_ids = []
    
    for wavlm_utt, wav2vec_utt in zip(all_wavlm_utterances, all_wav2vec_utterances):
        batch_wavlm.append(wavlm_utt['wavlm'])
        batch_wav2vec.append(wav2vec_utt['wav2vec'])
        batch_q_type_ids.append(wavlm_utt['q_type_id'])
        
        text = wavlm_utt.get('text', '')
        ttr_features = extract_enhanced_ttr_features(text)
        
        ttr_vector = [
            ttr_features['ttr'], ttr_features['ttr_log'],
            ttr_features['repetition_rate'], ttr_features['unique_word_ratio'],
            ttr_features['lexical_density'], ttr_features['word_length_variance']
        ]
        batch_ttr_enhanced.append(ttr_vector)
    
    batch_wavlm = torch.FloatTensor(np.array(batch_wavlm))
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))
    batch_ttr_enhanced = torch.FloatTensor(np.array(batch_ttr_enhanced))
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)
    
    return {
        'batch_wavlm': batch_wavlm,
        'batch_wav2vec': batch_wav2vec,
        'batch_ttr_enhanced': batch_ttr_enhanced,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }


# =============================================================================
# 데이터 로드
# =============================================================================
def load_raw_data():
    """원시 데이터 로드"""
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중...")
    print(f"{'='*70}")
    
    with open(WAVLM_DATA_PATH, 'rb') as f:
        wavlm_dataset = pickle.load(f)
    
    with open(WAV2VEC_DATA_PATH, 'rb') as f:
        wav2vec_dataset = pickle.load(f)
    
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset2.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    return wavlm_dataset, wav2vec_dataset, meta_df


def get_data_splits(wavlm_dataset, wav2vec_dataset, meta_df):
    """Train+Val / Test 분할"""
    common_pids = set(wavlm_dataset.keys()) & set(wav2vec_dataset.keys())
    
    train_val_pids = [pid for pid in meta_df[meta_df['Group'].isin(['Train', 'Validation'])]['Participant_ID'].tolist() 
                      if pid in common_pids]
    test_pids = [pid for pid in meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
                 if pid in common_pids]
    
    train_val_labels = [wavlm_dataset[pid]['label'] for pid in train_val_pids]
    test_labels = [wavlm_dataset[pid]['label'] for pid in test_pids]
    
    print(f"\n데이터 분할:")
    print(f"  Train+Val: {len(train_val_pids)} (Normal: {train_val_labels.count(0)}, Depression: {train_val_labels.count(1)})")
    print(f"  Test: {len(test_pids)} (Normal: {test_labels.count(0)}, Depression: {test_labels.count(1)})")
    
    return train_val_pids, test_pids


# =============================================================================
# 평가 함수
# =============================================================================
def evaluate_model(model, dataloader, criterion, threshold=0.5):
    model.eval()
    all_labels = []
    all_probs = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                batch_q_type_ids, batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0
    
    return all_labels, all_probs, avg_loss


def compute_metrics(labels, probs, threshold=0.5):
    """메트릭 계산"""
    preds = (np.array(probs) > threshold).astype(int)
    
    if len(np.unique(preds)) < 2:
        return {
            'f1_depression': 0.0, 'f1_normal': 0.0, 'balanced_f1': 0.0,
            'precision': 0.0, 'recall': 0.0, 'depression_focused_score': 0.0
        }
    
    f1_per_class = f1_score(labels, preds, average=None, zero_division=0)
    f1_normal = f1_per_class[0] if len(f1_per_class) > 0 else 0.0
    f1_depression = f1_per_class[1] if len(f1_per_class) > 1 else 0.0
    
    if f1_normal > 0 and f1_depression > 0:
        balanced_f1 = 2 * (f1_normal * f1_depression) / (f1_normal + f1_depression)
    else:
        balanced_f1 = 0.0
    
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    
    # Depression-focused score
    min_normal_f1 = 0.65
    if f1_normal >= min_normal_f1:
        depression_focused_score = f1_depression
    else:
        penalty = f1_normal / min_normal_f1
        depression_focused_score = f1_depression * penalty
    
    return {
        'f1_depression': f1_depression,
        'f1_normal': f1_normal,
        'balanced_f1': balanced_f1,
        'precision': precision,
        'recall': recall,
        'depression_focused_score': depression_focused_score
    }


def find_optimal_threshold_depression_focused(labels, probs, min_normal_f1=0.65):
    """Depression F1을 최대화하는 threshold 탐색"""
    best_score = 0.0
    best_threshold = 0.5
    best_metrics = None
    
    for thresh in np.arange(0.1, 0.9, 0.01):
        metrics = compute_metrics(labels, probs, thresh)
        score = metrics['depression_focused_score']
        
        if score > best_score:
            best_score = score
            best_threshold = thresh
            best_metrics = metrics.copy()
            best_metrics['threshold'] = thresh
    
    if best_metrics is None:
        best_metrics = {'threshold': 0.5, 'f1_depression': 0.0, 'f1_normal': 0.0, 
                       'balanced_f1': 0.0, 'depression_focused_score': 0.0}
    
    return best_threshold, best_metrics


# =============================================================================
# Test-Time Augmentation (TTA)
# =============================================================================
def evaluate_with_tta(model, dataloader, n_augmentations=5, dropout_rate=0.1):
    """
    Test-Time Augmentation: Dropout을 활성화하여 여러 번 예측 후 평균
    """
    model.train()  # Dropout 활성화
    
    all_labels = []
    all_probs_list = [[] for _ in range(n_augmentations)]
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wavlm = batch['batch_wavlm'].to(DEVICE)
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            if len(all_labels) == 0:
                all_labels.extend(batch_labels.cpu().numpy().flatten())
            elif len(all_labels) < len(batch_labels) + len(all_labels):
                all_labels.extend(batch_labels.cpu().numpy().flatten())
            
            for aug_idx in range(n_augmentations):
                logits, _ = model(
                    batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                    batch_q_type_ids, batch['num_utterances_list']
                )
                probs = torch.sigmoid(logits)
                all_probs_list[aug_idx].extend(probs.cpu().numpy().flatten())
    
    # 재수집 (labels)
    all_labels = []
    for batch in dataloader:
        all_labels.extend(batch['batch_labels'].numpy().flatten())
    
    # 확률 평균
    all_probs_array = np.array(all_probs_list)
    ensemble_probs = np.mean(all_probs_array, axis=0)
    
    model.eval()
    
    return all_labels, ensemble_probs


# =============================================================================
# K-Fold Cross-Validation 학습
# =============================================================================
def train_kfold_ensemble(wavlm_dataset, wav2vec_dataset, train_val_pids, test_pids, best_params, n_folds=5):
    """K-Fold Cross-Validation으로 앙상블 모델 학습"""
    
    print(f"\n{'='*70}")
    print(f"🔄 {n_folds}-Fold Cross-Validation Ensemble Training")
    print(f"{'='*70}")
    print(f"Parameters: {best_params}")
    
    # Train+Val 데이터셋 생성
    full_dataset = MultimodalUtteranceDataset(wavlm_dataset, wav2vec_dataset, pids=train_val_pids)
    labels = full_dataset.get_labels()
    
    # Test 데이터셋
    test_dataset = MultimodalUtteranceDataset(wavlm_dataset, wav2vec_dataset, pids=test_pids)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=multimodal_collate_fn, num_workers=0)
    
    # Stratified K-Fold
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_models = []
    fold_thresholds = []
    fold_val_results = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(range(len(full_dataset)), labels)):
        print(f"\n{'='*50}")
        print(f"📊 Fold {fold+1}/{n_folds}")
        print(f"{'='*50}")
        
        set_seed(42 + fold * 100)
        
        # Subset 생성
        train_subset = Subset(full_dataset, train_idx)
        val_subset = Subset(full_dataset, val_idx)
        
        train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                                 collate_fn=multimodal_collate_fn, num_workers=0)
        val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                               collate_fn=multimodal_collate_fn, num_workers=0)
        
        train_labels = [labels[i] for i in train_idx]
        val_labels = [labels[i] for i in val_idx]
        print(f"  Train: {len(train_idx)} (Dep: {sum(train_labels)})")
        print(f"  Val: {len(val_idx)} (Dep: {sum(val_labels)})")
        
        # 모델 생성
        model = WavLMCentricAuxiliaryFusion(
            d_model=256,
            nhead=8,
            num_encoder_layers=3,
            dim_feedforward=512,
            dropout=best_params.get('dropout', 0.3),
            aux_compression_ratio=best_params.get('aux_compression_ratio', 4)
        ).to(DEVICE)
        
        criterion = FocalLoss(
            alpha=best_params.get('focal_alpha', 0.7),
            gamma=best_params.get('focal_gamma', 2.0),
            label_smoothing=best_params.get('label_smoothing', 0.1)
        )
        
        optimizer = optim.AdamW(
            model.parameters(),
            lr=best_params.get('lr', 1e-4),
            weight_decay=best_params.get('weight_decay', 1e-4)
        )
        
        warmup_epochs = best_params.get('warmup_epochs', 5)
        warmup_scheduler = optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
        )
        cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-6
        )
        scheduler = optim.lr_scheduler.SequentialLR(
            optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs]
        )
        
        best_score = 0.0
        best_threshold = 0.5
        patience_counter = 0
        best_state = None
        
        for epoch in range(NUM_EPOCHS):
            # Training
            model.train()
            
            pbar = tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}")
            for batch in pbar:
                batch_wavlm = batch['batch_wavlm'].to(DEVICE)
                batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
                batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
                batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
                batch_labels_tensor = batch['batch_labels'].to(DEVICE)
                
                optimizer.zero_grad()
                logits, _ = model(
                    batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                    batch_q_type_ids, batch['num_utterances_list']
                )
                
                loss = criterion(logits, batch_labels_tensor)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
                optimizer.step()
                
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
            scheduler.step()
            
            # Validation
            val_labels_eval, val_probs, _ = evaluate_model(model, val_loader, criterion)
            opt_threshold, val_metrics = find_optimal_threshold_depression_focused(val_labels_eval, val_probs)
            
            current_score = val_metrics['depression_focused_score']
            
            if current_score > best_score:
                best_score = current_score
                best_threshold = opt_threshold
                patience_counter = 0
                best_state = deepcopy(model.state_dict())
                print(f"    Epoch {epoch+1}: Dep F1={val_metrics['f1_depression']:.4f}, "
                      f"Normal F1={val_metrics['f1_normal']:.4f} ✓")
            else:
                patience_counter += 1
                if patience_counter >= EARLY_STOPPING_PATIENCE:
                    print(f"    Early stopping at epoch {epoch+1}")
                    break
        
        # Best model 로드
        if best_state is not None:
            model.load_state_dict(best_state)
        
        fold_models.append(model)
        fold_thresholds.append(best_threshold)
        fold_val_results.append({
            'fold': fold + 1,
            'val_depression_f1': val_metrics['f1_depression'],
            'val_normal_f1': val_metrics['f1_normal'],
            'threshold': best_threshold
        })
        
        print(f"\n  Fold {fold+1} Best: Dep F1={val_metrics['f1_depression']:.4f}")
    
    return fold_models, fold_thresholds, fold_val_results, test_loader


# =============================================================================
# 앙상블 평가
# =============================================================================
def evaluate_ensemble(fold_models, test_loader, method='average'):
    """여러 fold 모델의 앙상블 평가"""
    
    print(f"\n{'='*70}")
    print(f"🎯 Ensemble Evaluation (Method: {method})")
    print(f"{'='*70}")
    
    all_labels = []
    all_probs_list = []
    
    # 각 모델의 예측 수집
    for fold_idx, model in enumerate(fold_models):
        model.eval()
        fold_probs = []
        
        with torch.no_grad():
            for batch in test_loader:
                batch_wavlm = batch['batch_wavlm'].to(DEVICE)
                batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
                batch_ttr_enhanced = batch['batch_ttr_enhanced'].to(DEVICE)
                batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
                
                if fold_idx == 0:
                    all_labels.extend(batch['batch_labels'].numpy().flatten())
                
                logits, _ = model(
                    batch_wavlm, batch_wav2vec, batch_ttr_enhanced,
                    batch_q_type_ids, batch['num_utterances_list']
                )
                probs = torch.sigmoid(logits)
                fold_probs.extend(probs.cpu().numpy().flatten())
        
        all_probs_list.append(fold_probs)
    
    # 앙상블 방법
    all_probs_array = np.array(all_probs_list)
    
    if method == 'average':
        ensemble_probs = np.mean(all_probs_array, axis=0)
    elif method == 'median':
        ensemble_probs = np.median(all_probs_array, axis=0)
    elif method == 'max':
        ensemble_probs = np.max(all_probs_array, axis=0)
    else:
        ensemble_probs = np.mean(all_probs_array, axis=0)
    
    return all_labels, ensemble_probs, all_probs_list


def evaluate_ensemble_with_tta(fold_models, test_loader, n_tta=5):
    """앙상블 + TTA"""
    
    print(f"\n{'='*70}")
    print(f"🎯 Ensemble + TTA Evaluation (TTA augmentations: {n_tta})")
    print(f"{'='*70}")
    
    all_labels = []
    all_probs_list = []
    
    for fold_idx, model in enumerate(fold_models):
        # TTA로 평가
        labels, tta_probs = evaluate_with_tta(model, test_loader, n_augmentations=n_tta)
        
        if fold_idx == 0:
            all_labels = labels
        
        all_probs_list.append(tta_probs)
    
    # 앙상블 (평균)
    ensemble_probs = np.mean(all_probs_list, axis=0)
    
    return all_labels, ensemble_probs


# =============================================================================
# 메인 실행
# =============================================================================
def run_kfold_ensemble_training(best_params=None):
    """K-Fold 앙상블 학습 및 평가 실행"""
    
    # 기본 파라미터 (Optuna에서 찾은 값 사용 가능)
    if best_params is None:
        best_params = {
            'focal_alpha': 0.7,
            'focal_gamma': 2.0,
            'label_smoothing': 0.1,
            'dropout': 0.3,
            'aux_compression_ratio': 4,
            'lr': 1e-4,
            'weight_decay': 1e-4,
            'warmup_epochs': 5
        }
    
    # 데이터 로드
    wavlm_dataset, wav2vec_dataset, meta_df = load_raw_data()
    train_val_pids, test_pids = get_data_splits(wavlm_dataset, wav2vec_dataset, meta_df)
    
    # K-Fold 학습
    fold_models, fold_thresholds, fold_val_results, test_loader = train_kfold_ensemble(
        wavlm_dataset, wav2vec_dataset, train_val_pids, test_pids, 
        best_params, n_folds=N_FOLDS
    )
    
    # ==========================================
    # 평가 방법 1: 단순 앙상블 (Average)
    # ==========================================
    labels, ensemble_probs_avg, individual_probs = evaluate_ensemble(fold_models, test_loader, method='average')
    opt_thresh_avg, metrics_avg = find_optimal_threshold_depression_focused(labels, ensemble_probs_avg)
    
    print(f"\n📊 Results - Ensemble (Average):")
    print(f"  Threshold: {opt_thresh_avg:.3f}")
    print(f"  Depression F1: {metrics_avg['f1_depression']:.4f} ⭐")
    print(f"  Normal F1: {metrics_avg['f1_normal']:.4f}")
    print(f"  Balanced F1: {metrics_avg['balanced_f1']:.4f}")
    
    # ==========================================
    # 평가 방법 2: 앙상블 (Median)
    # ==========================================
    _, ensemble_probs_median, _ = evaluate_ensemble(fold_models, test_loader, method='median')
    opt_thresh_med, metrics_med = find_optimal_threshold_depression_focused(labels, ensemble_probs_median)
    
    print(f"\n📊 Results - Ensemble (Median):")
    print(f"  Threshold: {opt_thresh_med:.3f}")
    print(f"  Depression F1: {metrics_med['f1_depression']:.4f} ⭐")
    print(f"  Normal F1: {metrics_med['f1_normal']:.4f}")
    print(f"  Balanced F1: {metrics_med['balanced_f1']:.4f}")
    
    # ==========================================
    # 평가 방법 3: 앙상블 + TTA
    # ==========================================
    labels_tta, ensemble_probs_tta = evaluate_ensemble_with_tta(fold_models, test_loader, n_tta=5)
    opt_thresh_tta, metrics_tta = find_optimal_threshold_depression_focused(labels_tta, ensemble_probs_tta)
    
    print(f"\n📊 Results - Ensemble + TTA:")
    print(f"  Threshold: {opt_thresh_tta:.3f}")
    print(f"  Depression F1: {metrics_tta['f1_depression']:.4f} ⭐")
    print(f"  Normal F1: {metrics_tta['f1_normal']:.4f}")
    print(f"  Balanced F1: {metrics_tta['balanced_f1']:.4f}")
    
    # ==========================================
    # 개별 Fold 결과
    # ==========================================
    print(f"\n📊 Individual Fold Results on Test Set:")
    for fold_idx, fold_probs in enumerate(individual_probs):
        opt_thresh, metrics = find_optimal_threshold_depression_focused(labels, fold_probs)
        print(f"  Fold {fold_idx+1}: Dep F1={metrics['f1_depression']:.4f}, "
              f"Normal F1={metrics['f1_normal']:.4f}, Thresh={opt_thresh:.3f}")
    
    # ==========================================
    # 최종 결과 선택
    # ==========================================
    all_results = {
        'average': {'metrics': metrics_avg, 'threshold': opt_thresh_avg, 'probs': ensemble_probs_avg},
        'median': {'metrics': metrics_med, 'threshold': opt_thresh_med, 'probs': ensemble_probs_median},
        'tta': {'metrics': metrics_tta, 'threshold': opt_thresh_tta, 'probs': ensemble_probs_tta}
    }
    
    # 가장 좋은 방법 선택
    best_method = max(all_results.keys(), key=lambda x: all_results[x]['metrics']['f1_depression'])
    best_result = all_results[best_method]
    
    print(f"\n{'='*70}")
    print(f"🏆 Best Method: {best_method.upper()}")
    print(f"{'='*70}")
    print(f"  Depression F1: {best_result['metrics']['f1_depression']:.4f} ⭐")
    print(f"  Normal F1: {best_result['metrics']['f1_normal']:.4f}")
    print(f"  Balanced F1: {best_result['metrics']['balanced_f1']:.4f}")
    print(f"  Optimal Threshold: {best_result['threshold']:.3f}")
    
    # Classification Report
    best_preds = (best_result['probs'] > best_result['threshold']).astype(int)
    print(f"\n분류 보고서:")
    print(classification_report(labels, best_preds, target_names=['Normal', 'Depression'], digits=4))
    
    # Confusion Matrix
    cm = confusion_matrix(labels, best_preds)
    print(f"Confusion Matrix:")
    print(f"  TN={cm[0,0]:3d}  FP={cm[0,1]:3d}")
    print(f"  FN={cm[1,0]:3d}  TP={cm[1,1]:3d}")
    
    # 모델 저장
    save_path = os.path.join(BASE_PATH, 'kfold_ensemble_models.pt')
    torch.save({
        'fold_models': [m.state_dict() for m in fold_models],
        'fold_thresholds': fold_thresholds,
        'fold_val_results': fold_val_results,
        'best_params': best_params,
        'best_method': best_method,
        'test_results': {
            'average': metrics_avg,
            'median': metrics_med,
            'tta': metrics_tta
        }
    }, save_path)
    print(f"\n✅ Models saved to: {save_path}")
    
    # JSON 결과 저장
    json_results = {
        'best_params': best_params,
        'fold_val_results': fold_val_results,
        'test_results': {
            'average': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                       for k, v in metrics_avg.items()},
            'median': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                      for k, v in metrics_med.items()},
            'tta': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                   for k, v in metrics_tta.items()}
        },
        'best_method': best_method
    }
    
    json_path = os.path.join(BASE_PATH, 'kfold_ensemble_results.json')
    with open(json_path, 'w') as f:
        json.dump(json_results, f, indent=2)
    print(f"✅ Results saved to: {json_path}")
    
    return fold_models, all_results


# =============================================================================
# 메인
# =============================================================================
if __name__ == "__main__":
    print(f"\n{'='*70}")
    print(f"🤖 K-Fold Ensemble + TTA Training")
    print(f"   Method: {N_FOLDS}-Fold CV + Test-Time Augmentation")
    print(f"{'='*70}\n")
    
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    
    # Optuna에서 찾은 최적 파라미터가 있으면 여기에 입력
    # 없으면 None으로 기본값 사용
    best_params = {
        'focal_alpha': 0.7,
        'focal_gamma': 2.0,
        'label_smoothing': 0.1,
        'dropout': 0.3,
        'aux_compression_ratio': 4,
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'warmup_epochs': 5
    }
    
    # 실행
    fold_models, results = run_kfold_ensemble_training(best_params)
    
    print(f"\n{'='*70}")
    print(f"✅ Training Complete!")
    print(f"{'='*70}\n")


🤖 K-Fold Ensemble + TTA Training
   Method: 5-Fold CV + Test-Time Augmentation

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
📂 데이터 로드 중...

데이터 분할:
  Train+Val: 139 (Normal: 96, Depression: 43)
  Test: 45 (Normal: 31, Depression: 14)

🔄 5-Fold Cross-Validation Ensemble Training
Parameters: {'focal_alpha': 0.7, 'focal_gamma': 2.0, 'label_smoothing': 0.1, 'dropout': 0.3, 'aux_compression_ratio': 4, 'lr': 0.0001, 'weight_decay': 0.0001, 'warmup_epochs': 5}

📊 Fold 1/5
  Train: 111 (Dep: 35)
  Val: 28 (Dep: 8)


Fold 1 Epoch 1: 100%|██████████| 14/14 [00:01<00:00,  9.91it/s, loss=0.0770]


    Epoch 1: Dep F1=0.3636, Normal F1=0.5882 ✓


Fold 1 Epoch 2: 100%|██████████| 14/14 [00:00<00:00, 14.09it/s, loss=0.0758]


    Epoch 2: Dep F1=0.4615, Normal F1=0.8372 ✓


...

    Epoch 12: Dep F1=0.7778, Normal F1=0.8889 ✓


Fold 5 Epoch 19: 100%|██████████| 14/14 [00:01<00:00, 13.18it/s, loss=0.0303]


    Epoch 19: Dep F1=0.8235, Normal F1=0.9189 ✓


Fold 5 Epoch 34: 100%|██████████| 14/14 [00:01<00:00, 12.72it/s, loss=0.0614]


    Early stopping at epoch 34

  Fold 5 Best: Dep F1=0.8235

🎯 Ensemble Evaluation (Method: average)

📊 Results - Ensemble (Average):
  Threshold: 0.580
  Depression F1: 0.6286 ⭐
  Normal F1: 0.7636
  Balanced F1: 0.6896

🎯 Ensemble Evaluation (Method: median)

📊 Results - Ensemble (Median):
  Threshold: 0.560
  Depression F1: 0.6111 ⭐
  Normal F1: 0.7407
  Balanced F1: 0.6697

🎯 Ensemble + TTA Evaluation (TTA augmentations: 5)

📊 Results - Ensemble + TTA:
  Threshold: 0.520
  Depression F1: 0.6286 ⭐
  Normal F1: 0.7636
  Balanced F1: 0.6896

📊 Individual Fold Results on Test Set:
  Fold 1: Dep F1=0.5882, Normal F1=0.7500, Thresh=0.520
  Fold 2: Dep F1=0.6111, Normal F1=0.7407, Thresh=0.510
  Fold 3: Dep F1=0.6000, Normal F1=0.6800, Thresh=0.480
  Fold 4: Dep F1=0.6061, Normal F1=0.7719, Thresh=0.620
  Fold 5: Dep F1=0.6286, Normal F1=0.7636, Thresh=0.600

🏆 Best Method: AVERAGE
  Depression F1: 0.6286 ⭐
  Normal F1: 0.7636
  Balanced F1: 0.6896
  Optimal Threshold: 0.580

분류 보고서:
   